# Smart Campus — Notebook Kelompok 2
**UTS DSA · Kelompok 2**

Notebook kerja untuk pencarian data, antrean prioritas, pemilihan ruangan, dan penjadwalan kampus.
Jalankan sel dari atas ke bawah.

Setup, impor/validasi Excel, Heap/PQ, Hash, Trie, DP, generator, pembanding jadwal, dan operasi dasar AVL sudah tersedia.
Penghapusan AVL, integrasi, 18 pengujian, dan eksperimen sudah dilengkapi; lihat bagian 6–10 untuk hasil aktual dan batas simulasi.


## 0.1 Pembagian kerja dan tujuan
| Anggota | Tanggung jawab |
|---|---|
| Ahmad Zacky | Koordinasi dan integrasi |
| Ikin Solihin + Deni Romadhon | Heap/PQ; Deni juga bagian layanan |
| Adit Praditia | Hash Table dan impor data |
| Mohammad Zilan | Trie dan pengujian |
| Akhmad Wildan + Asep Nurhidayat | AVL; eksperimen dan tampilan |
| Rifqi Ikhsan Rizkillah + Abdur Rozaq | DP dan pembanding semua kombinasi |

Alur penjelasan setiap anggota: **masalah → tujuan → operasi → alasan algoritma → contoh → batasan**.

| Tujuan | Pilihan algoritma | Hasil |
|---|---|---|
| Menangani permintaan penting dahulu | Min-Heap + Priority Queue | Urutan permintaan |
| Mengambil data dengan kode lengkap | Hash Table dengan chaining | Detail ruang/kelas/pasangan kode |
| Mencari nama yang baru diketik sebagian | Trie | Daftar saran nama |
| Mengurutkan pemakaian dan mencari kapasitas cukup | Dua AVL | Ranking dan calon ruang |
| Memilih kegiatan tanpa bentrok | Pemrograman Dinamis | Jadwal dan total nilai terbaik |

Struktur inti ditulis sendiri mengikuti rencana PPT. Jangan memakai `heapq` sebagai
implementasi Heap atau `dict` sebagai pengganti Hash Table. Dictionary boleh dipakai
untuk konfigurasi dan laporan. Anak Trie dan peta posisi Heap dapat memakai Hash Table
buatan tim. Streamlit menjadi tahap UAS; notebook ini tetap bisa dibuka tanpa memasangnya.


## 0.2 Aturan data dan asumsi
- Studi kasus: dua prodi memakai ruang F701–F708 dan F801–F808, mengikuti PPT.
- Slot kuliah reguler: 07.15–09.45, 10.15–12.45, 13.15–15.45; 15 slot per minggu.
- `priority`: makin kecil makin dahulu diproses. `value`: makin besar makin menambah nilai DP.
- Waktu berupa menit sejak 00.00. Interval `[start, end)` berarti dua kegiatan boleh bersebelahan.
- Pemakaian ruang = slot terpakai / 15 untuk demo ranking mingguan.
- **Kapasitas, peserta, dosen, dan isi kelas contoh merupakan data ilustrasi.**
  Data asli FEB dibaca terpisah pada bagian 0.4. Jumlah dari Excel dihitung saat impor;
  angka 118 kode/237 sesi pada PPT belum menjadi hasil pemeriksaan dataset ini.
- Kode pasangan yang disebut PPT dicatat eksplisit; jangan menebak pasangan dengan mengganti digit.
- Nilai 10/7/5/2 mengikuti PPT dan merupakan parameter rancangan, bukan kebijakan resmi kampus.
- PPT belum menetapkan lama tunggu untuk peningkatan prioritas. `AGING_STEP_MINUTES = None`
  berarti fitur belum diaktifkan; angka pengujian dapat ditetapkan sebagai asumsi terpisah.


In [1]:
from dataclasses import dataclass, replace
from typing import Optional
from random import Random
from itertools import combinations
from time import perf_counter
from statistics import median
import csv
import tracemalloc

PRIORITY = {"Examination": 1, "Practicum": 2, "Regular Class": 3, "Event": 4}
VALUE = {"Examination": 10, "Practicum": 7, "Regular Class": 5, "Event": 2}
WEEKLY_SLOTS = 15
AGING_STEP_MINUTES = None
DEMO_DAY = "2026-10-08"  # Hari contoh, bukan hasil impor jadwal resmi.
SEED = 42

def minutes(hhmm: str) -> int:
    hour, minute = map(int, hhmm.replace(".", ":").split(":"))
    if not (0 <= hour < 24 and 0 <= minute < 60):
        raise ValueError("Waktu harus berada dalam 00.00–23.59.")
    return hour * 60 + minute

REGULAR_SLOTS = [(minutes(a), minutes(b)) for a, b in
                 [("07:15", "09:45"), ("10:15", "12:45"), ("13:15", "15:45")]]

@dataclass(frozen=True)
class Room:
    room_id: str
    capacity: int
    room_type: str = "Classroom"
    used_slots: int = 0
    @property
    def utilization(self):
        return self.used_slots / WEEKLY_SLOTS

@dataclass(frozen=True)
class Course:
    course_code: str
    name: str
    language: str  # INT atau NAS

@dataclass(frozen=True)
class ClassRecord:
    class_code: str
    course_code: str
    participants: int
    lecturer_id: str
    room_id: str
    day: str
    start: int
    end: int
    activity_type: str = "Regular Class"

@dataclass(frozen=True)
class SearchEntry:
    entry_id: str
    name: str
    kind: str  # course atau lecturer

@dataclass(frozen=True)
class ScheduleRequest:
    request_id: str
    title: str
    activity_type: str
    participants: int
    room_id: str
    day: str
    start: int
    end: int
    value: int
    submitted_at: int = 0  # Menit pada jam simulasi antrean; terpisah dari jam kegiatan.
    @property
    def priority(self):
        return PRIORITY[self.activity_type]

def demo_or_todo(name, action):
    try:
        return action()
    except NotImplementedError:
        print(f"{name}: belum diimplementasikan — lengkapi TODO terlebih dahulu.")

print("Setup struktur data siap. Impor Excel menggunakan pandas dan openpyxl.")


Setup struktur data siap. Impor Excel menggunakan pandas dan openpyxl.


## 0.3 Data contoh bersama
Nama ruang dan contoh kegiatan mengikuti PPT. Kapasitas ditambahkan sebagai ilustrasi
agar pencarian 35 peserta bisa diuji. Angka pemakaian dibuat dari jumlah slot bulat,
sehingga 4/15, 7/15, 8/15, 9/15, 12/15, 14/15 membulat menjadi 27%, 47%, 53%, 60%, 80%, 93%.

Pasangan nama Indonesia–Inggris berikut adalah contoh alias, bukan salinan master data FEB.


In [2]:
room_ids = [f"F{floor}{number:02d}" for floor in (7, 8) for number in range(1, 9)]
rooms = [Room(rid, 60) for rid in room_ids]
room_overrides = {
    "F708": Room("F708", 30, used_slots=4),
    "F705": Room("F705", 40, used_slots=7),
    "F701": Room("F701", 45, used_slots=8),
    "F802": Room("F802", 60, used_slots=9),
    "F807": Room("F807", 80, used_slots=12),
    "F805": Room("F805", 100, used_slots=14),
}
rooms = [room_overrides.get(r.room_id, r) for r in rooms]

courses = [
    Course("MGT92119", "Principles of Management", "INT"),
    Course("MGT12119", "Dasar-Dasar Manajemen", "NAS"),
    Course("ACC93105", "Cost Accounting", "INT"),
    Course("ACC13101", "Akuntansi Biaya", "NAS"),
    Course("ACC93106", "Management Accounting", "INT"),
    Course("ACC13102", "Akuntansi Manajerial", "NAS"),
    Course("DEMO-AUD", "Audit Kecurangan", "NAS"),
    Course("DEMO-EKO", "Ekonomi Mikro", "NAS"),
]
equivalent_pairs = [("MGT92119", "MGT12119"), ("ACC93105", "ACC13101"),
                    ("ACC93106", "ACC13102")]
lecturers = [SearchEntry("DOS-01", "Dosen Anonim A", "lecturer"),
             SearchEntry("DOS-02", "Dosen Anonim B", "lecturer")]
search_entries = [SearchEntry(c.course_code, c.name, "course") for c in courses] + lecturers
class_records = [
    ClassRecord("DEMO-K01", "MGT92119", 15, "DOS-01", "F705", DEMO_DAY,
                minutes("10:15"), minutes("12:45")),
    ClassRecord("DEMO-K02", "MGT12119", 20, "DOS-01", "F705", DEMO_DAY,
                minutes("10:15"), minutes("12:45")),
]
# Enam kegiatan pada slide 15. Semua ditawarkan ke ruang F802 pada hari contoh.
activity_rows = [
    ("J1", "Seminar HIMA", "Event", "07:15", "08:15", 20),
    ("J2", "Kelas pengganti 1 SKS", "Regular Class", "08:15", "09:05", 25),
    ("J3", "UTS Akuntansi Manajerial", "Examination", "07:15", "09:15", 35),
    ("J4", "Sosialisasi KRS", "Event", "09:15", "10:15", 30),
    ("J5", "UTS LAB Pengantar Akuntansi", "Practicum", "10:15", "11:45", 35),
    ("J6", "Kelas pengganti 2 SKS", "Regular Class", "11:00", "12:40", 25),
]
requests = [ScheduleRequest(rid, title, kind, people, "F802", DEMO_DAY,
                           minutes(start), minutes(end), VALUE[kind])
            for rid, title, kind, start, end, people in activity_rows]
candidate_schedules = list(requests)

assert len(rooms) == 16 and len({r.room_id for r in rooms}) == 16
assert all(r.capacity > 0 and 0 <= r.used_slots <= WEEKLY_SLOTS for r in rooms)
assert len({c.course_code for c in courses}) == len(courses)
assert all(0 <= r.start < r.end <= 1440 and r.participants > 0 for r in requests)
assert all(r.value == VALUE[r.activity_type] for r in requests)
print(f"Data demo siap: {len(rooms)} ruang, {len(courses)} mata kuliah, {len(requests)} kegiatan.")
print("Target antrean: J3, J5, J2, J6, J1, J4. Target DP: J3 + J4 + J5 = 19.")


Data demo siap: 16 ruang, 8 mata kuliah, 6 kegiatan.
Target antrean: J3, J5, J2, J6, J1, J4. Target DP: J3 + J4 + J5 = 19.


## 0.4 Impor dan Validasi Data Asli
Jalankan sel berikut secara berurutan. Excel diambil dari Drive; bila diminta, masuk
ke akun Google yang memiliki akses. Untuk Jupyter, letakkan Excel di folder notebook.

**Sumber:** [dataset_jadwal_uts_feb_uph.xlsx](https://drive.google.com/file/d/14ssmxLs2leJDNAUBvTpuk2ukbxm8b-iP/view),
berdasarkan PDF **JADWAL UTS_TERM 1261_MANAJEMEN_DOSEN.pdf** dan
**JADWAL UTS_TERM 1261_AKUNTANSI_DOSEN.pdf**. Ini jadwal ujian 5–9 Oktober 2026.

- `data_asli`: isi empat tabel Excel, sebelum perapian.
- `data_bersih`: tanggal, jam, angka, dan teks yang sudah dirapikan.
- `laporan_masalah`: baris yang perlu diperiksa, beserta asalnya.

Baris bermasalah tetap disimpan. Durasi tertulis dan durasi berdasarkan jam ditampilkan
terpisah. Kapasitas ruang merupakan **asumsi**, dan pasangan INT–NAS belum menjadi
pasangan resmi. Beberapa kelas memang memakai ruang dan jam yang sama; hal itu
tidak langsung dianggap bentrok. Hasil impor belum digunakan untuk pemesanan ruang otomatis.

In [3]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, date, time
import re
import urllib.request
import zipfile
import hashlib
import pandas as pd

BERKAS_DATA = Path("dataset_jadwal_uts_feb_uph.xlsx")
ID_DATA_DRIVE = "14ssmxLs2leJDNAUBvTpuk2ukbxm8b-iP"
UNDUH_ULANG = False  # Ubah ke True untuk mengambil isi terbaru dari Drive.

def ambil_berkas_data(path, file_id, unduh_ulang=False):
    path = Path(path)
    if path.is_file() and not unduh_ulang:
        return path
    try:
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        with urllib.request.urlopen(url, timeout=30) as response:
            content = response.read()
        if not zipfile.is_zipfile(BytesIO(content)):
            raise ValueError("Unduhan belum berupa Excel.")
    except Exception:
        try:
            from google.colab import auth
        except ImportError as exc:
            raise FileNotFoundError(
                "Letakkan Excel di folder notebook, lalu jalankan sel ini kembali."
            ) from exc
        auth.authenticate_user()
        import google.auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        credentials, _ = google.auth.default()
        service = build("drive", "v3", credentials=credentials)
        buffer = BytesIO()
        downloader = MediaIoBaseDownload(buffer, service.files().get_media(fileId=file_id))
        selesai = False
        while not selesai:
            _, selesai = downloader.next_chunk()
        content = buffer.getvalue()
        if not zipfile.is_zipfile(BytesIO(content)):
            raise ValueError("File Drive bukan Excel yang dapat dibaca.")
    path.write_bytes(content)
    return path

KOLOM_WAJIB = {
    "jadwal_uts": ["hari", "mulai", "selesai", "durasi_menit", "kode_mk", "nama_mk",
                   "prodi_mk", "tipe_kelas", "jenis_sesi", "sks", "kode_kelas", "angkatan",
                   "peserta", "sifat_ujian", "mode_ujian", "ruang", "dosen_id", "pengawas_id",
                   "keterangan", "prodi_dokumen"],
    "mata_kuliah": ["kode_mk", "nama_mk", "tipe_kelas", "prodi_mk", "jumlah_kelas"],
    "pasangan_int_nas": ["kode_mk_int", "nama_int", "kode_mk_nas", "nama_nas", "metode", "status"],
    "ruang": ["ruang", "gedung", "lantai", "sesi_ujian", "peserta_maks_per_kelas",
              "peserta_maks_per_slot", "kapasitas_asumsi"],
}

def import_feb_data(excel_path):
    with pd.ExcelFile(excel_path, engine="openpyxl") as excel:
        kurang = set(KOLOM_WAJIB) - set(excel.sheet_names)
        if kurang:
            raise ValueError(f"Lembar Excel belum lengkap: {sorted(kurang)}")
        tables = {}
        for nama, wajib in KOLOM_WAJIB.items():
            # Hindari pembacaan "09.30" sebagai angka 9.3 yang menghilangkan nol.
            tabel = pd.read_excel(excel, sheet_name=nama, dtype=object)
            kurang = set(wajib) - set(tabel.columns)
            if kurang:
                raise ValueError(f"Kolom pada {nama} belum lengkap: {sorted(kurang)}")
            # Indeks asal dipertahankan agar nomor baris Excel dapat ditelusuri.
            tables[nama] = tabel.dropna(how="all").copy(deep=True)
    return tables

BERKAS_DATA = ambil_berkas_data(BERKAS_DATA, ID_DATA_DRIVE, UNDUH_ULANG)
data_asli = import_feb_data(BERKAS_DATA)
jejak_sumber = {
    "nama_file": BERKAS_DATA.name,
    "sha256": hashlib.sha256(BERKAS_DATA.read_bytes()).hexdigest(),
    "id_drive": ID_DATA_DRIVE,
}
print("Excel berhasil dibaca:")
for nama, tabel in data_asli.items():
    print(f"- {nama}: {len(tabel)} baris")

Excel berhasil dibaca:
- jadwal_uts: 243 baris
- mata_kuliah: 122 baris
- pasangan_int_nas: 61 baris
- ruang: 16 baris


### 0.4.1 Rapikan Format dan Periksa Kelengkapan
Contoh: `Senin, 05 Okt 2026` menjadi tanggal `2026-10-05`, dan `07.15` menjadi
435 menit sejak tengah malam. Angka peserta tetap bilangan bulat, sedangkan kode
kelas dan mata kuliah tetap teks. Teks dirapikan tanpa mengubah nama atau kode.

Kolom `durasi_menit_sumber` menyimpan angka tertulis; `durasi_dari_jam` dihitung
dari jam mulai dan selesai. Selisihnya perlu diperiksa, bukan dipilih otomatis.
Ruang kosong untuk ujian **Online** diperbolehkan. Identitas dosen dan pengawas
menggunakan kode anonim yang sudah tersedia pada Excel.

In [4]:
BULAN = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "jun": 6,
         "jul": 7, "agu": 8, "ags": 8, "sep": 9, "okt": 10, "nov": 11, "des": 12}
HARI = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]

def rapikan_teks(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, str):
        hasil = re.sub(r"\s+", " ", value).strip()
        return hasil if hasil else pd.NA
    return value

def baca_tanggal(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (datetime, date, pd.Timestamp)):
        return pd.Timestamp(value).normalize()
    teks = str(value).split(",")[-1].strip()
    try:
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", teks):
            return pd.Timestamp(date.fromisoformat(teks))
        cocok = re.fullmatch(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", teks)
        if cocok:
            d, bulan, y = cocok.groups()
            return pd.Timestamp(date(int(y), BULAN[bulan.lower()], int(d)))
    except (ValueError, KeyError):
        pass
    return pd.NaT

def baca_jam(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (datetime, time)):
        return value.hour * 60 + value.minute if value.second == 0 else pd.NA
    cocok = re.fullmatch(r"(\d{1,2})[.:](\d{2})", str(value).strip())
    if cocok:
        jam, menit = map(int, cocok.groups())
        if 0 <= jam <= 23 and 0 <= menit <= 59:
            return jam * 60 + menit
    return pd.NA

# Hasil pembandingan PDF; nomor halaman mengikuti urutan halaman dokumen.
PERBEDAAN_RUANG_PDF = pd.DataFrame([
    ["2217/BIKM", "F704", 1, "F708", 1],
    ["2185/BIKM", "F804", 3, "F703", 1],
    ["2226/AIKM", "F805", 3, "F708", 2],
], columns=["kode_kelas", "ruang_manajemen", "halaman_manajemen",
            "ruang_akuntansi", "halaman_akuntansi"])

def rapikan_dan_validasi(data, perbedaan_ruang=PERBEDAAN_RUANG_PDF):
    bersih, masalah = {}, []

    def catat(tabel, mask, jenis, kolom, detail):
        mask = pd.Series(mask, index=tabel.index).fillna(False).astype(bool)
        for _, row in tabel.loc[mask].iterrows():
            masalah.append({
                "lembar": row["lembar_sumber"], "baris_excel": row["baris_excel"],
                "kode": row.get("kode_kelas", row.get("kode_mk_int", row.get("kode_mk", row.get("ruang")))),
                "jenis": jenis, "kolom": kolom, "detail": detail,
            })

    def bilangan_bulat(tabel, nama, minimum=0):
        angka = pd.to_numeric(tabel[nama], errors="coerce")
        sah = angka.notna() & angka.ge(minimum) & angka.mod(1).eq(0)
        # Nilai yang gagal dibaca ditandai, tidak diganti menjadi nol.
        catat(tabel, ~sah, "angka_tidak_valid", nama, f"Harus bilangan bulat minimal {minimum}.")
        tabel[nama] = angka.where(sah).astype("Int64")

    for nama, asli in data.items():
        tabel = asli.copy(deep=True)
        for kolom in tabel.columns:
            if tabel[kolom].dtype == "object":
                tabel[kolom] = tabel[kolom].map(rapikan_teks)
        tabel["lembar_sumber"] = nama
        tabel["baris_excel"] = tabel.index + 2
        tabel["berkas_sumber"] = "dataset_jadwal_uts_feb_uph.xlsx"
        bersih[nama] = tabel

    jadwal = bersih["jadwal_uts"]
    for nama in ["kode_kelas", "kode_mk", "nama_mk", "dosen_id", "mode_ujian", "jenis_sesi"]:
        catat(jadwal, jadwal[nama].isna(), "kolom_kosong", nama, "Kolom ini perlu diisi.")
    for nama in ["kode_kelas", "kode_mk", "ruang", "dosen_id", "pengawas_id"]:
        jadwal[nama] = jadwal[nama].astype("string")
    for nama, minimal in [("peserta", 1), ("durasi_menit", 1), ("sks", 0), ("angkatan", 1)]:
        bilangan_bulat(jadwal, nama, minimal)
    jadwal["durasi_menit_sumber"] = jadwal["durasi_menit"].copy()
    jadwal["tanggal"] = pd.to_datetime(jadwal["hari"].map(baca_tanggal))
    jadwal["mulai_menit"] = jadwal["mulai"].map(baca_jam).astype("Int64")
    jadwal["selesai_menit"] = jadwal["selesai"].map(baca_jam).astype("Int64")
    jadwal["durasi_dari_jam"] = jadwal["selesai_menit"] - jadwal["mulai_menit"]
    jadwal["hari_baku"] = jadwal["tanggal"].map(lambda x: HARI[x.weekday()] if pd.notna(x) else pd.NA)
    catat(jadwal, jadwal["tanggal"].isna(), "tanggal_tidak_valid", "hari", "Tanggal tidak terbaca.")
    punya_nama_hari = jadwal["hari"].astype("string").str.contains(",", na=False)
    nama_hari = jadwal["hari"].astype("string").str.split(",").str[0].str.strip().str.casefold()
    catat(jadwal, punya_nama_hari & nama_hari.ne(jadwal["hari_baku"].astype("string").str.casefold()),
          "nama_hari_berbeda", "hari", "Nama hari berbeda dengan tanggal kalender.")
    jam_valid = jadwal[["mulai_menit", "selesai_menit"]].notna().all(axis=1)
    catat(jadwal, ~jam_valid, "jam_tidak_valid", "mulai/selesai", "Jam harus dalam format HH.MM atau HH:MM.")
    catat(jadwal, jam_valid & jadwal["durasi_dari_jam"].le(0), "urutan_jam_tidak_valid",
          "mulai/selesai", "Jam selesai harus lebih besar dari jam mulai.")
    catat(jadwal, jam_valid & jadwal["durasi_dari_jam"].gt(0)
          & jadwal["durasi_dari_jam"].ne(jadwal["durasi_menit_sumber"]),
          "durasi_berbeda", "durasi_menit", "Durasi tertulis berbeda dengan selisih jam.")
    mode = jadwal["mode_ujian"].astype("string").str.casefold()
    catat(jadwal, ~mode.isin(["online", "onsite"]), "mode_tidak_valid", "mode_ujian", "Mode harus Online atau Onsite.")
    catat(jadwal, mode.eq("onsite") & jadwal["ruang"].isna(),
          "ruang_kosong", "ruang", "Ujian Onsite perlu memiliki ruang.")
    catat(jadwal, mode.eq("online") & jadwal["ruang"].notna(),
          "ruang_ujian_online", "ruang", "Periksa alasan ruang terisi pada ujian Online.")
    catat(jadwal, jadwal.duplicated("kode_kelas", keep=False) & jadwal["kode_kelas"].notna(),
          "kode_kelas_berulang", "kode_kelas", "Periksa sesi atau duplikasi; baris tetap disimpan.")

    ruang = bersih["ruang"]
    ruang["ruang"] = ruang["ruang"].astype("string")
    catat(ruang, ruang["ruang"].isna(), "kolom_kosong", "ruang", "Kode ruang perlu diisi.")
    catat(ruang, ruang.duplicated("ruang", keep=False), "kode_ruang_berulang", "ruang", "Kode ruang harus unik.")
    for nama in ["lantai", "sesi_ujian", "peserta_maks_per_kelas", "peserta_maks_per_slot"]:
        bilangan_bulat(ruang, nama)
    bilangan_bulat(ruang, "kapasitas_asumsi", 1)
    ruang["status_kapasitas"] = "asumsi; belum kapasitas resmi"
    catat(jadwal, jadwal["ruang"].notna() & ~jadwal["ruang"].isin(ruang["ruang"]),
          "ruang_tidak_dikenal", "ruang", "Kode ruang tidak terdapat pada tabel ruang.")

    mk = bersih["mata_kuliah"]
    mk["kode_mk"] = mk["kode_mk"].astype("string")
    for nama in ["kode_mk", "nama_mk"]:
        catat(mk, mk[nama].isna(), "kolom_kosong", nama, "Kolom ini perlu diisi.")
    bilangan_bulat(mk, "jumlah_kelas", 1)
    catat(mk, mk.duplicated("kode_mk", keep=False), "kode_mk_berulang", "kode_mk", "Kode mata kuliah harus unik.")
    catat(jadwal, jadwal["kode_mk"].notna() & ~jadwal["kode_mk"].isin(mk["kode_mk"]),
          "kode_mk_tidak_dikenal", "kode_mk", "Kode tidak terdapat pada tabel mata kuliah.")

    pasangan = bersih["pasangan_int_nas"]
    for nama in ["kode_mk_int", "kode_mk_nas"]:
        pasangan[nama] = pasangan[nama].astype("string")
        catat(pasangan, pasangan[nama].notna() & ~pasangan[nama].isin(mk["kode_mk"]),
              "kode_pasangan_tidak_dikenal", nama, "Kode tidak terdapat pada tabel mata kuliah.")
    catat(pasangan, pasangan["kode_mk_int"].isna(), "kolom_kosong", "kode_mk_int", "Kode INT perlu diisi.")
    perlu_cek = pasangan["status"].astype("string").str.casefold().ne("otomatis").fillna(True) | pasangan["kode_mk_nas"].isna()
    catat(pasangan, perlu_cek, "pasangan_perlu_periksa", "kode_mk_int/kode_mk_nas",
          "Periksa pasangan pada dokumen; kode NAS kosong tetap dibiarkan kosong.")
    # Aturan digit pada Excel merupakan dugaan, bukan bukti kesetaraan resmi.
    pasangan["status_verifikasi"] = "belum dikonfirmasi resmi"

    for _, beda in perbedaan_ruang.iterrows():
        mask = jadwal["kode_kelas"].eq(beda["kode_kelas"])
        detail = (f"PDF Manajemen hlm. {beda['halaman_manajemen']}: {beda['ruang_manajemen']}; "
                  f"PDF Akuntansi hlm. {beda['halaman_akuntansi']}: {beda['ruang_akuntansi']}. "
                  "Nilai Excel tetap disimpan; pastikan ruang yang berlaku.")
        catat(jadwal, mask, "ruang_berbeda_antar_pdf", "ruang", detail)

    laporan = pd.DataFrame(masalah, columns=["lembar", "baris_excel", "kode", "jenis", "kolom", "detail"])
    for nama, tabel in bersih.items():
        perlu_periksa = set(laporan.loc[laporan["lembar"].eq(nama), "baris_excel"])
        tabel["perlu_periksa"] = tabel["baris_excel"].isin(perlu_periksa)
    jadwal["kategori_kegiatan"] = "Ujian"  # Jenis sesi LAB tidak mengubah ujian menjadi praktikum.
    return bersih, laporan

data_bersih, laporan_masalah = rapikan_dan_validasi(data_asli)
jadwal_bersih = data_bersih["jadwal_uts"]
ruang_bersih = data_bersih["ruang"]
pasangan_bersih = data_bersih["pasangan_int_nas"]
print("Perapian selesai. Semua baris data tetap disimpan.")

Perapian selesai. Semua baris data tetap disimpan.


### 0.4.2 Hasil Pemeriksaan
Jumlah di bawah dihitung dari file yang dibaca, bukan angka contoh PPT.
Satu baris bisa memiliki lebih dari satu catatan. `perlu_periksa = False` berarti
lolos pemeriksaan format yang tersedia, bukan jaminan jadwal atau kapasitas sudah resmi.

Tiga perbedaan ruang berasal dari pembandingan dua PDF. Pemeriksaan ini memakai
daftar temuan PDF yang dicatat pada sel sebelumnya, sehingga perlu diperbarui bila
versi PDF berubah. Kelas yang memakai ruang bersamaan belum digabung otomatis.

In [5]:
def tampilkan_ringkasan(data, laporan):
    jadwal, mk, ruang, pasangan = (data[n] for n in
        ["jadwal_uts", "mata_kuliah", "ruang", "pasangan_int_nas"])
    onsite = jadwal[jadwal["mode_ujian"].astype("string").str.casefold().eq("onsite")]
    grup = onsite.groupby(["tanggal", "ruang", "mulai_menit", "selesai_menit"], dropna=False).size()
    ringkasan = pd.DataFrame([
        ["Baris jadwal", len(jadwal)], ["Kode mata kuliah unik", mk["kode_mk"].nunique()],
        ["Ruangan", ruang["ruang"].nunique()],
        ["Ujian Onsite", len(onsite)], ["Ujian Online", jadwal["mode_ujian"].astype("string").str.casefold().eq("online").sum()],
        ["Durasi berbeda", laporan["jenis"].eq("durasi_berbeda").sum()],
        ["Ruang berbeda antar-PDF", laporan["jenis"].eq("ruang_berbeda_antar_pdf").sum()],
        ["Pasangan perlu diperiksa", laporan["jenis"].eq("pasangan_perlu_periksa").sum()],
        ["Pasangan tanpa kode NAS", pasangan["kode_mk_nas"].isna().sum()],
        ["Kelompok ruang/jam dengan beberapa kelas", grup.gt(1).sum()],
    ], columns=["Pemeriksaan", "Jumlah"])
    print(ringkasan.to_string(index=False))
    print("\nJenis catatan data:")
    if laporan.empty:
        print("Tidak ditemukan masalah pada pemeriksaan yang tersedia.")
    else:
        print(laporan.groupby("jenis").size().rename("jumlah").to_string())
    print("\nKapasitas seluruh ruang masih asumsi; semua pasangan belum dikonfirmasi resmi.")
    return ringkasan

ringkasan_validasi = tampilkan_ringkasan(data_bersih, laporan_masalah)

                             Pemeriksaan  Jumlah
                            Baris jadwal     243
                   Kode mata kuliah unik     122
                                 Ruangan      16
                            Ujian Onsite     207
                            Ujian Online      36
                          Durasi berbeda       9
                 Ruang berbeda antar-PDF       3
                Pasangan perlu diperiksa       9
                 Pasangan tanpa kode NAS       6
Kelompok ruang/jam dengan beberapa kelas      47

Jenis catatan data:
jenis
durasi_berbeda             9
pasangan_perlu_periksa     9
ruang_berbeda_antar_pdf    3

Kapasitas seluruh ruang masih asumsi; semua pasangan belum dikonfirmasi resmi.


In [6]:
print("DURASI YANG PERLU DIPERIKSA")
baris_durasi = laporan_masalah.loc[
    laporan_masalah["jenis"].eq("durasi_berbeda"), "baris_excel"]
kolom_durasi = ["baris_excel", "kode_kelas", "tanggal", "mulai", "selesai",
                "durasi_menit_sumber", "durasi_dari_jam"]
print(jadwal_bersih.loc[jadwal_bersih["baris_excel"].isin(baris_durasi), kolom_durasi].to_string(index=False))

print("\nRUANG YANG BERBEDA ANTAR-PDF")
temuan_ruang = PERBEDAAN_RUANG_PDF.merge(
    jadwal_bersih[["kode_kelas", "ruang", "baris_excel"]], on="kode_kelas", how="inner")
print(temuan_ruang.rename(columns={"ruang": "ruang_excel"}).to_string(index=False))

print("\nPASANGAN YANG PERLU DIPERIKSA")
baris_pasangan = laporan_masalah.loc[
    laporan_masalah["jenis"].eq("pasangan_perlu_periksa"), "baris_excel"]
print(pasangan_bersih.loc[pasangan_bersih["baris_excel"].isin(baris_pasangan),
      ["baris_excel", "kode_mk_int", "nama_int", "kode_mk_nas", "metode", "status"]].to_string(index=False))

print("\nCONTOH DATA SETELAH DIRAPIKAN")
print(jadwal_bersih[["kode_kelas", "nama_mk", "tanggal", "mulai_menit", "selesai_menit",
                    "peserta", "ruang", "perlu_periksa"]].head(5).to_string(index=False))

DURASI YANG PERLU DIPERIKSA
 baris_excel kode_kelas    tanggal mulai selesai  durasi_menit_sumber  durasi_dari_jam
          57   2538/MAN 2026-10-05 13.15   15.15                  100              120
         107   2836/ARK 2026-10-07 10.15   12.15                  100              120
         108   2838/CRK 2026-10-07 10.15   12.15                  100              120
         109   2837/BRK 2026-10-07 10.15   12.15                  100              120
         116   2526/CRK 2026-10-07 13.15   14.45                  100               90
         125   2524/ARK 2026-10-07 13.15   14.45                  100               90
         126   2525/BRK 2026-10-07 13.15   14.45                  100               90
         190   2536/PJK 2026-10-09 07.15   08.45                  100               90
         206   2535/PJK 2026-10-09 09.30   11.00                  100               90

RUANG YANG BERBEDA ANTAR-PDF
kode_kelas ruang_manajemen  halaman_manajemen ruang_akuntansi  halaman_a

### 0.4.3 Sebelum Dipakai oleh Algoritma
Tim perlu memastikan durasi yang berlaku, memilih ruang untuk tiga kelas dengan
sumber berbeda, dan mengonfirmasi pasangan INT–NAS. Kapasitas resmi juga perlu
ditanyakan; angka pada Excel hanya perkiraan.

Untuk ujian, `sesi_ujian` pada tabel ruang menghitung baris kelas. Angka ini belum
dapat dipakai sebagai jumlah slot terisi. Penjadwalan perlu memperhitungkan kelas
yang berbagi ruang, serta lama ujian yang berbeda-beda.

Data contoh `rooms` dan `requests` tetap dipakai untuk latihan algoritma berikut.
Data asli tersedia terpisah dalam `data_bersih` sampai aturan tersebut disepakati.

### 0.4.4 Bahan Data untuk Pencarian
**Persiapan dan penghubung data: Zacky/Deni · Hash Table: Adit · Trie: Zilan.**

Sel berikut menyiapkan masukan dari Excel untuk modul anggota. Hash Table dan Trie
diisi pada bagian 2 dan 3. Daftar ini dapat dipakai setelah modul tersebut selesai.

| Bahan | Isi | Pemakaian oleh anggota |
|---|---|---|
| `bahan_pencarian_feb["kelas"]` | Pasangan kode kelas dan detail jadwal | Hash: `put(kode_kelas, detail)` |
| `bahan_pencarian_feb["mata_kuliah"]` | Pasangan kode MK, nama, dan daftar kode kelas | Hash: `put(kode_mk, detail)` |
| `bahan_pencarian_feb["nama"]` | Nama asli/variasi nama untuk kode MK yang sama | Trie nama: `insert(nama, kode_mk)` |
| `bahan_pencarian_feb["kode"]` | Kode MK sebagai teks dan ID hasil | Trie kode: `insert(kode_mk, kode_mk)` |
| `jadwal_tampil_feb` | Tabel detail untuk ditampilkan | Cocokkan kode hasil dengan baris kelas |

Nama yang sama boleh memiliki beberapa kode. Variasi nama pada **kode yang sama**
tetap disimpan; pasangan INT–NAS belum digabung. Kode hasil Trie perlu ditampilkan
sekali walaupun beberapa variasi nama cocok. Nama asli ditampilkan, sementara
pencarian memakai `strip().casefold()` sesuai kontrak Trie.

Catatan data ikut dibawa ke hasil pencarian. Daftar ini tidak mengalokasikan ruang
atau memasukkan ujian ke antrean penjadwalan.
Tipe kelas dipetakan menjadi `INT` untuk Internasional dan `NAS` untuk Nasional;
nama tipe dari sumber juga tetap disimpan dalam detail data.

In [7]:
def nilai_biasa(value):
    """Ubah nilai pandas menjadi nilai Python; data kosong tetap None."""
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, "item") else value

def jam_tampil(value):
    if pd.isna(value):
        return None
    value = int(value)
    return f"{value // 60:02d}:{value % 60:02d}"

def tipe_kelas_singkat(value):
    if pd.isna(value):
        return None
    pemetaan = {"internasional": "INT", "nasional": "NAS", "int": "INT", "nas": "NAS"}
    return pemetaan.get(str(value).strip().casefold(), str(value))

def siapkan_bahan_pencarian_feb(data, laporan):
    jadwal = data["jadwal_uts"]
    katalog = data["mata_kuliah"]
    kelas, mata_kuliah, nama, kode = [], [], [], []
    tabel_tampil = []

    for _, row in jadwal.iterrows():
        catatan = laporan.loc[
            laporan["lembar"].eq("jadwal_uts") & laporan["baris_excel"].eq(row["baris_excel"])]
        daftar_catatan = (catatan["jenis"] + ": " + catatan["detail"]).tolist()
        tanggal = row["tanggal"].strftime("%Y-%m-%d") if pd.notna(row["tanggal"]) else None
        detail = {
            "class_code": nilai_biasa(row["kode_kelas"]),
            "course_code": nilai_biasa(row["kode_mk"]),
            "name": nilai_biasa(row["nama_mk"]),
            "language": tipe_kelas_singkat(row["tipe_kelas"]),
            "language_source": nilai_biasa(row["tipe_kelas"]),
            "program": nilai_biasa(row["prodi_mk"]),
            "participants": nilai_biasa(row["peserta"]),
            "lecturer_id": nilai_biasa(row["dosen_id"]),
            "room_id": nilai_biasa(row["ruang"]),
            "day": tanggal,
            "start": nilai_biasa(row["mulai_menit"]),
            "end": nilai_biasa(row["selesai_menit"]),
            "exam_mode": nilai_biasa(row["mode_ujian"]),
            "session_type": nilai_biasa(row["jenis_sesi"]),
            "duration_source": nilai_biasa(row["durasi_menit_sumber"]),
            "duration_from_time": nilai_biasa(row["durasi_dari_jam"]),
            "needs_review": bool(row["perlu_periksa"]),
            "review_notes": daftar_catatan,
            "source": {"sheet": row["lembar_sumber"], "excel_row": int(row["baris_excel"]),
                       "source_programs": nilai_biasa(row["prodi_dokumen"])},
        }
        kelas.append((detail["class_code"], detail))
        tabel_tampil.append({
            "Kode MK": detail["course_code"], "Mata kuliah": detail["name"],
            "Kode kelas": detail["class_code"], "Prodi": detail["program"],
            "INT/NAS": detail["language"], "Tanggal": tanggal,
            "Jenis sesi": detail["session_type"],
            "Mulai": jam_tampil(row["mulai_menit"]), "Selesai": jam_tampil(row["selesai_menit"]),
            "Peserta": detail["participants"], "Mode": detail["exam_mode"], "Ruang": detail["room_id"],
            "Perlu diperiksa": "Ya" if detail["needs_review"] else "Tidak",
            "Catatan": " | ".join(daftar_catatan), "Baris Excel": detail["source"]["excel_row"],
        })

    # Gabungkan hanya nama dan kelas pada kode MK yang persis sama.
    kode_semua = pd.concat([katalog["kode_mk"], jadwal["kode_mk"]]).dropna().drop_duplicates()
    for course_code in sorted(kode_semua.tolist()):
        dari_katalog = katalog.loc[katalog["kode_mk"].eq(course_code)]
        dari_jadwal = jadwal.loc[jadwal["kode_mk"].eq(course_code)]
        variasi_nama = pd.concat([dari_katalog["nama_mk"], dari_jadwal["nama_mk"]]).dropna().drop_duplicates().tolist()
        baris_dasar = dari_katalog.iloc[0] if not dari_katalog.empty else dari_jadwal.iloc[0]
        detail = {
            "course_code": str(course_code), "name": nilai_biasa(baris_dasar["nama_mk"]),
            "language": tipe_kelas_singkat(baris_dasar["tipe_kelas"]),
            "language_source": nilai_biasa(baris_dasar["tipe_kelas"]),
            "program": nilai_biasa(baris_dasar["prodi_mk"]),
            "class_codes": dari_jadwal["kode_kelas"].dropna().tolist(),
            "name_variants": variasi_nama,
        }
        mata_kuliah.append((str(course_code), detail))
        for variasi in variasi_nama:
            nama.append(SearchEntry(str(course_code), variasi, "course"))
        kode.append(SearchEntry(str(course_code), str(course_code), "course"))

    bahan = {"kelas": kelas, "mata_kuliah": mata_kuliah, "nama": nama, "kode": kode}
    return bahan, pd.DataFrame(tabel_tampil)

bahan_pencarian_feb, jadwal_tampil_feb = siapkan_bahan_pencarian_feb(data_bersih, laporan_masalah)
print("Bahan data siap untuk modul anggota:")
print(f"- {len(bahan_pencarian_feb['kelas'])} detail kelas untuk Hash Table")
print(f"- {len(bahan_pencarian_feb['mata_kuliah'])} kode mata kuliah untuk Hash Table")
print(f"- {len(bahan_pencarian_feb['nama'])} entri nama/variasi nama untuk Trie")
print(f"- {len(bahan_pencarian_feb['kode'])} entri kode untuk Trie")
print("Kelas belum digabung; catatan durasi dan ruang tetap disertakan.")

Bahan data siap untuk modul anggota:
- 243 detail kelas untuk Hash Table
- 122 kode mata kuliah untuk Hash Table
- 137 entri nama/variasi nama untuk Trie
- 122 entri kode untuk Trie
Kelas belum digabung; catatan durasi dan ruang tetap disertakan.


### 0.4.5 Contoh Hasil yang Diharapkan
Tabel berikut diambil langsung dari bahan data untuk menjadi patokan saat anggota
menguji modulnya. Ini **contoh keluaran**, belum hasil menjalankan Hash Table/Trie.

| Masukan | Makna |
|---|---|
| `ACC93105` | Kode lengkap: tampilkan mata kuliah dan semua kelasnya |
| `akun` | Awalan nama: tampilkan kode yang memiliki nama berawalan “akun” |
| `ACC9` | Awalan kode: tampilkan kode berawalan “ACC9” |

Nama dan kode asli tetap ditampilkan. Hasil awalan tidak menambahkan pasangan
INT–NAS otomatis. Bila kode tidak ditemukan, keluaran yang disepakati adalah
daftar kosong dengan pesan **“Kode tidak ditemukan.”**

In [8]:
# Contoh tetap dari sumber untuk membandingkan hasil modul anggota nanti.
contoh_kode_lengkap = jadwal_tampil_feb.loc[jadwal_tampil_feb["Kode MK"].eq("ACC93105")].copy()
contoh_awalan_nama = pd.DataFrame([
    {"Kode MK": entry.entry_id, "Nama yang cocok": entry.name}
    for entry in bahan_pencarian_feb["nama"]
    if entry.name.strip().casefold().startswith("akun")
], columns=["Kode MK", "Nama yang cocok"]).drop_duplicates("Kode MK").sort_values("Kode MK")
contoh_awalan_kode = pd.DataFrame([
    {"Kode MK": key, "Mata kuliah": detail["name"], "Jumlah kelas": len(detail["class_codes"])}
    for key, detail in bahan_pencarian_feb["mata_kuliah"]
    if key.strip().casefold().startswith("acc9")
], columns=["Kode MK", "Mata kuliah", "Jumlah kelas"]).sort_values("Kode MK")

print("CONTOH KODE LENGKAP: ACC93105")
print(contoh_kode_lengkap[["Kode MK", "Mata kuliah", "Kode kelas", "INT/NAS", "Jenis sesi", "Tanggal",
      "Mulai", "Selesai", "Peserta", "Mode", "Ruang", "Perlu diperiksa"]].to_string(index=False))
print(f"\nCONTOH AWALAN NAMA: akun ({len(contoh_awalan_nama)} kode; ditampilkan maksimal 5)")
print(contoh_awalan_nama.head(5).to_string(index=False))
print(f"\nCONTOH AWALAN KODE: ACC9 ({len(contoh_awalan_kode)} kode; ditampilkan maksimal 5)")
print(contoh_awalan_kode.head(5).to_string(index=False))
print("\nBahan data dan contoh keluaran siap. Pencarian Hash Table/Trie menunggu modul anggota.")

CONTOH KODE LENGKAP: ACC93105
 Kode MK                   Mata kuliah Kode kelas INT/NAS Jenis sesi    Tanggal Mulai Selesai  Peserta   Mode Ruang Perlu diperiksa
ACC93105   Lab Accounting Principles 1  2468/AILA     INT        LAB 2026-10-09 07:15   08:45        7 Onsite  F707           Tidak
ACC93105 Accounting Principles 1 + LAB  2467/CIKM     INT      Teori 2026-10-09 07:15   09:15       26 Onsite  F806           Tidak
ACC93105 Accounting Principles 1 + LAB  2466/BIKM     INT      Teori 2026-10-09 07:15   09:15       29 Onsite  F807           Tidak
ACC93105 Accounting Principles 1 + LAB  2465/AIKA     INT      Teori 2026-10-09 13:15   15:15        7 Onsite  F707           Tidak
ACC93105   Lab Accounting Principles 1  2470/CILM     INT        LAB 2026-10-09 13:15   14:45       26 Onsite  F806           Tidak
ACC93105   Lab Accounting Principles 1  2469/BILM     INT        LAB 2026-10-09 13:15   14:45       29 Onsite  F807           Tidak

CONTOH AWALAN NAMA: akun (5 kode; ditampilkan

### 0.5 Generator Percobaan
Generator membuat data acak dengan seed tetap agar percobaan dapat diulang.
Data ini belum mengikuti distribusi jadwal FEB. Gunakan jumlah kecil saat belajar;
100 ribu data digunakan setelah implementasi dan pengukuran siap.

In [9]:
def generate_rooms(n, seed=SEED):
    if not isinstance(n, int) or n < 0:
        raise ValueError("Jumlah data harus bilangan bulat tidak negatif.")
    rng = Random(seed)
    return [Room(f"SYN-R{i:06d}", rng.choice([30, 40, 60, 80, 100]),
                 used_slots=rng.randrange(WEEKLY_SLOTS + 1)) for i in range(n)]

def generate_requests(n, seed=SEED):
    if not isinstance(n, int) or n < 0:
        raise ValueError("Jumlah data harus bilangan bulat tidak negatif.")
    rng = Random(seed)
    data = []
    for i in range(n):
        kind = rng.choice(list(PRIORITY))
        duration = rng.choice([60, 75, 90, 100, 120, 150])
        start = rng.randrange(435, 960 - duration + 1)
        data.append(ScheduleRequest(f"SYN-J{i:06d}", f"Kegiatan sintetis {i}", kind,
                                    rng.randrange(10, 61), "F802", DEMO_DAY,
                                    start, start + duration, VALUE[kind], i))
    return data

def generate_search_entries(n):
    # Dua awalan untuk eksperimen yang menghasilkan sedikit/banyak jawaban.
    return [SearchEntry(f"SYN-C{i:06d}", f"{('Akuntansi' if i % 2 else 'Manajemen')} {i:06d}",
                        "course") for i in range(n)]

print("Generator siap. Data sintetis tidak dianggap jadwal resmi FEB.")


Generator siap. Data sintetis tidak dianggap jadwal resmi FEB.


## 1. Heap + Priority Queue — Prioritas Permintaan
**Ikin + Deni · PPT 7, 8, 12**

Tujuan: ambil permintaan yang paling perlu ditangani berikutnya tanpa mengurutkan
ulang semua data. Simpan entri dengan key **`(prioritas, jam_mulai, urutan_masuk)`**.
Ujian=1, praktikum=2, kuliah=3, kegiatan lain=4. Jika prioritas sama, jam lebih awal
didahulukan; FIFO hanya berlaku jika **prioritas dan jam mulai sama**.

Gunakan array sendiri: parent `(i-1)//2`, anak `2*i+1` dan `2*i+2`.
`push` menukar ke atas; `pop` memindahkan elemen terakhir lalu menukar ke bawah.
Heap menjaga hubungan orang tua–anak; arraynya tidak harus terurut penuh.

Peningkatan prioritas karena menunggu: `max(1, prioritas_awal - waktu_tunggu // T)`.
T belum ditetapkan PPT. Simpan prioritas awal, waktu masuk, dan counter asli;
perbarui **key yang benar-benar dibandingkan**, bukan hanya atribut permintaan.
Jika memakai peta posisi, perbarui posisi setiap pertukaran dan penghapusan.
Saat fitur ini aktif, permintaan lama dapat menyamai prioritas ujian.

- Push/pop: O(log n); peek: O(1); build lewat sift-down: O(n); ruang: O(n).
- Mengubah satu key dengan posisi diketahui: O(log n).
- Mengunjungi semua permintaan untuk peningkatan prioritas: O(n), lalu membangun
  ulang heap O(n), atau O(n log n) bila mengubah setiap key secara terpisah.

**Terimplementasi:** push/pop/peek, build O(n), aging, FIFO, kasus kosong, Hash posisi, penghitung perbandingan, dan pemeriksaan invariant. Implementasi memakai array manual, tanpa heapq/sort. ID duplikat dan parameter tidak valid ditolak sebelum mutasi. Aging dihitung ulang dari prioritas awal; counter dan request asli tetap disimpan.

Heap mengacu ke HashTableChaining bagian 2. Karena itu demo dan tes dijalankan sesudah definisi Hash agar **Run all dari kernel baru** dapat berjalan. Fixture campuran kategori hanya untuk uji; UI memakai Excel FEB. Antrean UI dipisah per tanggal, lalu membandingkan jam mulai. Jam pengajuan/aging adalah simulasi terpisah, bukan waktu dari Excel.


In [10]:
class MinHeapPriorityQueue:
    """Min-heap manual: (prioritas efektif, jam mulai, urutan masuk).

    Antrean dipakai untuk satu tanggal kegiatan. Jam simulasi submitted_at/now
    terpisah dari jam mulai kegiatan. Hash tim menyimpan ID -> posisi array.
    """
    def __init__(self):
        self.entries = []  # (key, ScheduleRequest asli, counter asli)
        self.next_counter = 0
        self.position_map = HashTableChaining()
        self.comparisons = 0
        self.now = None
        self.aging_step = None

    def _check_request(self, request):
        if not isinstance(request, ScheduleRequest):
            raise TypeError("Masukan harus ScheduleRequest.")
        if not isinstance(request.request_id, str) or not request.request_id.strip():
            raise ValueError("ID permintaan harus teks tidak kosong.")
        if request.activity_type not in PRIORITY:
            raise ValueError("Kategori kegiatan tidak dikenal.")
        for name in ("start", "end", "submitted_at"):
            value = getattr(request, name)
            if type(value) is not int or value < 0:
                raise ValueError(f"{name} harus bilangan bulat tidak negatif.")
        if not 0 <= request.start < request.end <= 1440:
            raise ValueError("Jam kegiatan harus berurutan dalam satu hari.")
        if self.now is not None and request.submitted_at > self.now:
            raise ValueError("Waktu masuk tidak boleh melewati jam aging aktif.")

    def _key(self, request, counter):
        priority = request.priority
        if self.aging_step is not None:
            waited = self.now - request.submitted_at
            priority = max(1, priority - waited // self.aging_step)
        return (priority, request.start, counter)

    def _less(self, left, right):
        self.comparisons += 1
        return self.entries[left][0] < self.entries[right][0]

    def _swap(self, left, right):
        self.entries[left], self.entries[right] = self.entries[right], self.entries[left]
        self.position_map.put(self.entries[left][1].request_id, left)
        self.position_map.put(self.entries[right][1].request_id, right)

    def _sift_up(self, index):
        while index > 0:
            parent = (index - 1) // 2
            if not self._less(index, parent):
                break
            self._swap(index, parent)
            index = parent

    def _sift_down(self, index):
        while 2 * index + 1 < len(self.entries):
            child = 2 * index + 1
            right = child + 1
            if right < len(self.entries) and self._less(right, child):
                child = right
            if not self._less(child, index):
                break
            self._swap(index, child)
            index = child

    def _heapify(self):
        # Daun sudah heap; perbaiki hanya simpul internal dari bawah ke atas.
        for index in range(len(self.entries) // 2 - 1, -1, -1):
            self._sift_down(index)

    def build(self, requests):
        """Ganti antrean dengan batch baru dalam O(n), mempertahankan jam aging.

        Validasi seluruh batch sebelum mengubah antrean. Counter baru mengikuti
        urutan batch. ID duplikat ditolak memakai Hash buatan tim.
        """
        entries = []
        positions = HashTableChaining()
        for counter, request in enumerate(requests):
            self._check_request(request)
            if positions.contains(request.request_id):
                raise ValueError(f"ID berulang: {request.request_id}")
            positions.put(request.request_id, counter)
            entries.append((self._key(request, counter), request, counter))
        self.entries = entries
        self.position_map = positions
        self.next_counter = len(entries)
        self.comparisons = 0
        self._heapify()

    def push(self, request):
        self._check_request(request)
        if self.position_map.contains(request.request_id):
            raise ValueError(f"ID sudah berada dalam antrean: {request.request_id}")
        counter = self.next_counter
        self.entries.append((self._key(request, counter), request, counter))
        self.position_map.put(request.request_id, len(self.entries) - 1)
        self.next_counter += 1
        self._sift_up(len(self.entries) - 1)

    def pop(self):
        if not self.entries:
            raise IndexError("Antrean kosong.")
        request = self.entries[0][1]
        last = self.entries.pop()
        self.position_map.delete(request.request_id)
        if self.entries:
            self.entries[0] = last
            self.position_map.put(last[1].request_id, 0)
            self._sift_down(0)
        return request

    def peek(self):
        if not self.entries:
            raise IndexError("Antrean kosong.")
        return self.entries[0][1]

    def apply_aging(self, now, step_minutes):
        """Hitung ulang dari prioritas AWAL; panggilan ulang tidak menumpuk aging.

        Counter/FIFO dan objek request tidak berubah. O(n) untuk hitung key lalu
        heapify O(n). Jam simulasi harus maju/diam dan tidak mendahului pengajuan.
        """
        if type(step_minutes) is not int or step_minutes <= 0:
            raise ValueError("Interval aging harus bilangan bulat positif.")
        if type(now) is not int or now < 0:
            raise ValueError("Jam simulasi harus bilangan bulat tidak negatif.")
        if self.now is not None and now < self.now:
            raise ValueError("Jam simulasi tidak boleh mundur.")
        if any(request.submitted_at > now for _, request, _ in self.entries):
            raise ValueError("Jam simulasi mendahului waktu masuk permintaan.")
        self.now, self.aging_step = now, step_minutes
        self.entries = [(self._key(request, counter), request, counter)
                        for _, request, counter in self.entries]
        self._heapify()

    def ordered_entries(self):
        """Pratinjau O(n log n) lewat pop pada salinan; antrean asli tidak berubah."""
        copy = MinHeapPriorityQueue()
        copy.entries = list(self.entries)
        for index, (_, request, _) in enumerate(copy.entries):
            copy.position_map.put(request.request_id, index)
        result = []
        while copy:
            result.append(copy.entries[0])
            copy.pop()
        return result

    def validate(self):
        """Periksa heap, counter unik, key, dan Hash posisi tanpa menambah metrik."""
        assert len(self.position_map) == len(self.entries)
        assert len({counter for _, _, counter in self.entries}) == len(self.entries)
        for index, (key, request, counter) in enumerate(self.entries):
            assert self.position_map.get(request.request_id) == index
            assert key == self._key(request, counter)
            assert counter < self.next_counter
            if index > 0:
                assert self.entries[(index - 1) // 2][0] <= key
        return True

    def __len__(self):
        return len(self.entries)


def demo_heap():
    queue = MinHeapPriorityQueue()
    for request in requests:
        queue.push(request)
    result = [queue.pop().request_id for _ in requests]
    assert result == ["J3", "J5", "J2", "J6", "J1", "J4"]
    print("Urutan antrean:", result)


# HashTableChaining didefinisikan pada bagian 2. Demo dan tes Heap dijalankan
# setelah sel Hash agar Run all dari kernel baru tidak bergantung state lama.


## 2. Hash Table — Pencarian Berdasarkan Kode
**Adit · PPT 9, 12**

Buat tiga tabel: `room_id → Room`, `class_code → ClassRecord`, dan
`course_code → kode_pasangan`. Isi pasangan dua arah. Nama mata kuliah dapat diambil
dari katalog tambahan yang juga memakai class Hash Table yang sama.

Setiap bucket menyimpan chain pasangan key/value. `put` dengan key yang sudah ada
memperbarui value; `get` key yang tidak ada menghasilkan None; `delete` menghasilkan
True jika terhapus dan False jika tidak ditemukan. Bedakan None yang disimpan dari key
tidak ada dengan `contains` bila diperlukan.

Hash utama: `h = (31*h + ord(c)) % jumlah_bucket`. Hash penjumlahan hanya untuk
ilustrasi: F708 dan F807 sama-sama menuju bucket 5 ketika jumlah bucket 8.
Hash polynomial dapat memisahkannya, tetapi collision tetap mungkin.

Ambang 0,7 mengikuti PPT: gandakan bucket jika melewatinya. Satu proses membangun
ulang tabel memerlukan O(n + m) untuk n entri dan m bucket; biasanya m sebanding n.
Insert rata-rata O(1) secara amortisasi jika ukuran dijaga; worst-case O(n).
Biaya membaca key string sepanjang L juga perlu dihitung. Memori O(n + m).

**TODO:** chaining, update, delete, perubahan ukuran, validasi dan penghitung probe.


In [11]:
def weak_hash(key, bucket_count):
    return sum(map(ord, key)) % bucket_count

def polynomial_hash(key, bucket_count):
    h = 0
    for character in key:
        h = (h * 31 + ord(character)) % bucket_count
    return h

class HashTableChaining:
    def __init__(self, bucket_count=8, max_load=0.7):
        if bucket_count < 1 or max_load <= 0:
            raise ValueError("Jumlah bucket dan ambang harus positif.")
        self.buckets = [[] for _ in range(bucket_count)]
        self.max_load = max_load
        self.size = 0
        self.probes = 0

    def _hash(self, key):
        return polynomial_hash(key, len(self.buckets))

    def _check_key(self, key):
        if not isinstance(key, str):
            raise TypeError(
                f"Key harus berupa teks (str), bukan {type(key).__name__}.")

    def _find(self, chain, key):
        """Cari key di satu chain. Hasil: posisi di chain, atau -1 bila tidak ada."""
        for position, (stored_key, _) in enumerate(chain):
            self.probes += 1
            if stored_key == key:
                return position
        return -1

    def put(self, key, value):
        self._check_key(key)
        chain = self.buckets[self._hash(key)]
        position = self._find(chain, key)

        if position != -1:
            # Key sudah ada -> perbarui value saja, size tidak bertambah.
            chain[position] = (key, value)
            return

        # Key baru -> tambahkan ke chain.
        chain.append((key, value))
        self.size += 1

        # Cek kepenuhan hanya setelah INSERT BARU (bukan setelah update).
        if self.size / len(self.buckets) > self.max_load:
            self._rehash(len(self.buckets) * 2)

    def get(self, key):
        """Kembalikan value, atau None bila key tidak ada."""
        self._check_key(key)
        chain = self.buckets[self._hash(key)]
        position = self._find(chain, key)
        if position == -1:
            return None
        return chain[position][1]

    def contains(self, key):
        """True bila key ada. Dipakai untuk membedakan 'key tidak ada'
        dari 'key ada tetapi value-nya None'."""
        self._check_key(key)
        chain = self.buckets[self._hash(key)]
        return self._find(chain, key) != -1

    def delete(self, key):
        """True bila berhasil dihapus, False bila key tidak ditemukan."""
        self._check_key(key)
        chain = self.buckets[self._hash(key)]
        position = self._find(chain, key)
        if position == -1:
            return False
        chain.pop(position)
        self.size -= 1
        return True

    def items(self):
        """Hasilkan semua pasangan (key, value), laci demi laci."""
        for chain in self.buckets:
            for pair in chain:
                yield pair

    def _rehash(self, new_bucket_count):
        if new_bucket_count < 1:
            raise ValueError("Jumlah bucket baru harus positif.")
        # 1) Simpan dulu semua pasangan lama.
        old_pairs = list(self.items())
        # 2) Ganti laci dengan yang baru. Harus SEBELUM menghitung hash ulang,
        #    karena _hash memakai len(self.buckets).
        self.buckets = [[] for _ in range(new_bucket_count)]
        # 3) Masukkan lagi langsung ke chain. Tidak perlu cek duplikat
        #    (key lama sudah pasti unik) dan size tidak berubah.
        for key, value in old_pairs:
            self.buckets[self._hash(key)].append((key, value))

    def __len__(self):
        return self.size

    def __repr__(self):
        return f"HashTableChaining(size={self.size}, buckets={len(self.buckets)})"

def build_indexes():
    room_table, class_table = HashTableChaining(), HashTableChaining()
    equivalence_table, course_table = HashTableChaining(), HashTableChaining()
    for room in rooms:
        room_table.put(room.room_id, room)
    for record in class_records:
        class_table.put(record.class_code, record)
    for course in courses:
        course_table.put(course.course_code, course)
    for international, national in equivalent_pairs:
        equivalence_table.put(international, national)
        equivalence_table.put(national, international)
    return room_table, class_table, equivalence_table, course_table

assert weak_hash("F708", 8) == weak_hash("F807", 8) == 5
print("Collision ilustrasi: F708 dan F807 → bucket 5 (hash penjumlahan).")
demo_or_todo("Hash Table", build_indexes)


Collision ilustrasi: F708 dan F807 → bucket 5 (hash penjumlahan).


In [12]:
class WeakHashTable(HashTableChaining):
    """Khusus tes T05: memakai hash penjumlahan agar F708 dan F807 bertabrakan."""
    def _hash(self, key):
        return weak_hash(key, len(self.buckets))


def test_t04():
    # Put/get 10 ribu ID unik, semua data harus kembali benar.
    table = HashTableChaining()
    for i in range(10_000):
        table.put(f"ID-{i:05d}", i)
    assert len(table) == 10_000
    for i in range(10_000):
        assert table.get(f"ID-{i:05d}") == i
    assert table.get("ID-tidak-ada") is None


def test_t05():
    # F708 dan F807 bertabrakan (hash penjumlahan, 8 bucket) tapi tetap terbaca.
    table = WeakHashTable(bucket_count=8)
    table.put("F708", "ruang A")
    table.put("F807", "ruang B")
    assert len(table.buckets[5]) == 2          # keduanya satu chain di bucket 5
    assert table.get("F708") == "ruang A"
    assert table.get("F807") == "ruang B"


def test_t06():
    # Load factor melewati 0,7 -> jumlah bucket berubah, semua key tetap ada.
    table = HashTableChaining(bucket_count=8)
    keys = [f"KEY-{i:03d}" for i in range(100)]
    for n, key in enumerate(keys, start=1):
        table.put(key, n)
        if n == 5:
            assert len(table.buckets) == 8      # 5/8 = 0,625 -> belum resize
        if n == 6:
            assert len(table.buckets) == 16     # 6/8 = 0,75 > 0,7 -> resize
    assert len(table) == 100
    assert len(table.buckets) > 8
    for n, key in enumerate(keys, start=1):
        assert table.get(key) == n
    assert len(list(table.items())) == len(table)


def test_hash_tambahan():
    table = HashTableChaining()
    # update: size tidak bertambah, value berganti
    table.put("A", 1)
    table.put("A", 2)
    assert len(table) == 1 and table.get("A") == 2
    # delete: True bila ada, False bila tidak ada
    assert table.delete("A") is True
    assert table.delete("A") is False
    assert len(table) == 0 and table.get("A") is None
    # value None yang disimpan vs key tidak ada
    table.put("N", None)
    assert table.get("N") is None and table.contains("N")
    assert not table.contains("X")
    # key bukan teks ditolak
    for bad_key in (None, 123, 4.5):
        try:
            table.put(bad_key, "x")
        except TypeError:
            pass
        else:
            raise AssertionError("Key bukan teks seharusnya ditolak.")
    # delete lalu isi lagi, tetap benar
    table.put("B", 5)
    table.delete("B")
    table.put("B", 6)
    assert table.get("B") == 6
    # probes naik saat menelusuri chain
    before = table.probes
    table.get("B")
    assert table.probes > before


def run_hash_tests():
    for name, test in [("T04", test_t04), ("T05", test_t05),
                       ("T06", test_t06), ("Tambahan", test_hash_tambahan)]:
        test()
        print(f"{name}: LULUS")

run_hash_tests()

T04: LULUS
T05: LULUS
T06: LULUS
Tambahan: LULUS


### Uji Heap/PQ — Ikin & Deni
Dijalankan setelah Hash siap: demo J1–J6, T01–T03, aging, 1.000 operasi acak, build batch, dan validasi input. Data contoh berikut khusus pengujian, bukan data UI.


In [13]:
def heap_test_request(identifier, kind="Examination", start=600, submitted_at=0):
    # Fixture pengujian saja, bukan sumber jadwal aplikasi.
    return ScheduleRequest(identifier, identifier, kind, 20, "F802", "2026-10-08",
                           start, start + 60, VALUE[kind], submitted_at)


def test_t01():
    batch = [heap_test_request("E", "Event", 435),
             heap_test_request("K2", "Regular Class", 700),
             heap_test_request("U2", "Examination", 700),
             heap_test_request("P", "Practicum", 500),
             heap_test_request("U1", "Examination", 600),
             heap_test_request("K1", "Regular Class", 600)]
    queue = MinHeapPriorityQueue()
    for request in batch:
        queue.push(request)
        assert queue.validate()
    assert [queue.pop().request_id for _ in batch] == ["U1", "U2", "P", "K1", "K2", "E"]
    assert queue.validate()


def test_t02():
    queue = MinHeapPriorityQueue()
    batch = [heap_test_request("Masuk-1"), heap_test_request("Masuk-2"),
             heap_test_request("Jam-lebih-awal", start=500)]
    queue.build(batch)
    assert queue.peek() is batch[2] and len(queue) == 3
    assert [queue.pop().request_id for _ in batch] == ["Jam-lebih-awal", "Masuk-1", "Masuk-2"]


def test_t03():
    queue = MinHeapPriorityQueue()
    for action in (queue.pop, queue.peek):
        try:
            action()
        except IndexError:
            assert len(queue) == 0 and queue.validate()
        else:
            raise AssertionError("Antrean kosong harus menghasilkan IndexError.")


def test_heap_aging():
    event = heap_test_request("Event-lama", "Event", 500)
    exam = heap_test_request("Ujian-baru", "Examination", 600, 85)
    queue = MinHeapPriorityQueue()
    queue.build([event, exam])
    assert queue.peek() is exam
    counters = {r.request_id: c for _, r, c in queue.entries}
    queue.apply_aging(90, 30)
    assert queue.peek() is event and event.priority == 4
    once = list(queue.entries)
    queue.apply_aging(90, 30)
    assert queue.entries == once  # Tidak mengurangi prioritas lagi secara kumulatif.
    assert counters == {r.request_id: c for _, r, c in queue.entries}
    queue.push(heap_test_request("Baru", "Event", 435, 90))
    assert queue.entries[queue.position_map.get("Baru")][0][0] == 4
    for now, step in [(89, 30), (100, 0), (100, -1), (100, True), (100.5, 30)]:
        before = (list(queue.entries), queue.now, queue.aging_step)
        try:
            queue.apply_aging(now, step)
        except ValueError:
            assert (queue.entries, queue.now, queue.aging_step) == before
        else:
            raise AssertionError("Parameter aging tidak valid seharusnya ditolak.")
    assert queue.validate()
    queue.apply_aging(180, 30)
    assert [queue.pop().request_id for _ in range(len(queue))] == ["Baru", "Event-lama", "Ujian-baru"]


def test_heap_random():
    rng = Random(42)
    queue = MinHeapPriorityQueue()
    reference = []
    counter = 0
    for i in range(1000):
        if reference and rng.random() < 0.45:
            reference.sort(key=lambda item: (item[0].priority, item[0].start, item[1]))
            expected, _ = reference.pop(0)
            assert queue.peek() is expected
            assert queue.pop() is expected
        else:
            kind = rng.choice(list(PRIORITY))
            request = heap_test_request(f"ACAK-{i}", kind, rng.randrange(435, 900))
            queue.push(request)
            reference.append((request, counter))
            counter += 1
        assert queue.validate()
    expected = [r for r, c in sorted(reference, key=lambda item: (item[0].priority, item[0].start, item[1]))]
    before = (list(queue.entries), queue.comparisons, queue.next_counter)
    assert [entry[1] for entry in queue.ordered_entries()] == expected
    assert (queue.entries, queue.comparisons, queue.next_counter) == before
    assert [queue.pop() for _ in expected] == expected
    # Heapify batch, bukan sort; pembanding hanya digunakan sebagai oracle tes.
    batch = [heap_test_request(f"B-{i}", rng.choice(list(PRIORITY)), rng.randrange(435, 900)) for i in range(1000)]
    queue.build(batch)
    assert queue.validate() and queue.comparisons <= 2 * len(batch)
    expected = sorted(enumerate(batch), key=lambda item: (item[1].priority, item[1].start, item[0]))
    assert [queue.pop() for _ in batch] == [r for _, r in expected]


def test_heap_invalid():
    queue = MinHeapPriorityQueue()
    first = heap_test_request("A")
    queue.push(first)
    invalid = [first, replace(first, request_id=""), replace(first, activity_type="Unknown"),
               replace(first, start=700, end=650), replace(first, submitted_at=-1),
               replace(first, start=True)]
    for request in invalid:
        before = (list(queue.entries), queue.next_counter, queue.comparisons)
        try:
            queue.push(request)
        except ValueError:
            assert (queue.entries, queue.next_counter, queue.comparisons) == before
            assert queue.validate()
        else:
            raise AssertionError("Masukan tidak valid/ID berulang harus ditolak.")
    try:
        queue.build([heap_test_request("B"), heap_test_request("B")])
    except ValueError:
        assert len(queue) == 1 and queue.peek() is first and queue.validate()
    else:
        raise AssertionError("Batch dengan ID berulang harus ditolak tanpa mengubah antrean.")
    queue = MinHeapPriorityQueue()
    queue.push(heap_test_request("Masa-depan", submitted_at=100))
    try:
        queue.apply_aging(90, 30)
    except ValueError:
        assert queue.now is None and queue.aging_step is None and queue.validate()
    else:
        raise AssertionError("Jam aging mendahului waktu masuk harus ditolak.")


def run_heap_tests():
    for name, action in [("T01", test_t01), ("T02", test_t02), ("T03", test_t03),
                         ("Aging", test_heap_aging), ("Acak + build", test_heap_random),
                         ("Validasi", test_heap_invalid)]:
        action()
        print(f"Heap {name}: LULUS")


demo_heap()
run_heap_tests()


Urutan antrean: ['J3', 'J5', 'J2', 'J6', 'J1', 'J4']
Heap T01: LULUS
Heap T02: LULUS
Heap T03: LULUS
Heap Aging: LULUS
Heap Acak + build: LULUS
Heap Validasi: LULUS


## 3. Trie — Saran Nama dan Kode Mata Kuliah
**Zilan · PPT 10, 13**

Buat Trie nama (Indonesia, Inggris, dosen) dan Trie kode mata kuliah terpisah.
Normalisasi dengan `strip().casefold()`; simpan nama asli untuk ditampilkan.
Node akhir menyimpan beberapa ID agar nama yang sama tidak menimpa entri lain.

- Awalan **ak** → Akuntansi Biaya dan Akuntansi Manajerial.
- Awalan kode **ACC9** → ACC93105 dan ACC93106.
- Awalan **Dasar** langsung cocok dengan Dasar-Dasar Manajemen. Untuk ikut menampilkan
  Principles of Management, cari pasangan melalui tabel equivalence lalu ambil nama
  dari katalog. **Trie sendiri tidak menerjemahkan nama.** Hilangkan duplikasi hasil.
- Prefix kosong mengembalikan semua entri; prefix tidak ada menghasilkan daftar kosong.

Insert/exact search O(L), menemukan node prefix O(p). Mengumpulkan entri tersimpan:
O(p + s + k), dengan s node cabang yang dikunjungi dan k entri yang keluar.
Jika membuat ulang string hasil, biaya panjang string hasil juga ikut dihitung.
Memori mengikuti total node karakter dan ID terminal. Klaim “selalu sama cepat”
tidak berlaku ketika jumlah hasil meningkat. Biaya anak Hash Table bersifat rata-rata.

**TODO:** anak memakai Hash Table buatan tim, penanda akhir, DFS, alias dua bahasa, deduplikasi.


In [14]:
class TrieNode:
    def __init__(self):
        self.children = HashTableChaining()
        self.entry_ids = []
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
        self.visited_nodes = 0

    def insert(self, text, entry_id):
        if not isinstance(text, str):
            return
        normalized = text.strip().casefold()
        current = self.root
        for char in normalized:
            self.visited_nodes += 1
            if not current.children.contains(char):
                current.children.put(char, TrieNode())
            current = current.children.get(char)
        current.is_end = True
        if entry_id not in current.entry_ids:
            current.entry_ids.append(entry_id)

    def search(self, text):
        if not isinstance(text, str):
            return []
        normalized = text.strip().casefold()
        current = self.root
        for char in normalized:
            if not current.children.contains(char):
                return []
            current = current.children.get(char)
        if current.is_end:
            return list(current.entry_ids)
        return []

    def starts_with(self, prefix):
        if not isinstance(prefix, str):
            return False
        normalized = prefix.strip().casefold()
        current = self.root
        for char in normalized:
            if not current.children.contains(char):
                return False
            current = current.children.get(char)
        return True

    def autocomplete(self, prefix):
        if not isinstance(prefix, str):
            return []
        normalized = prefix.strip().casefold()
        current = self.root
        for char in normalized:
            if not current.children.contains(char):
                return []
            current = current.children.get(char)

        results = []
        def dfs(node):
            if node.is_end:
                for entry_id in node.entry_ids:
                    if entry_id not in results:
                        results.append(entry_id)
            for _, child_node in node.children.items():
                dfs(child_node)

        dfs(current)
        return results

def bilingual_autocomplete(trie, prefix, equivalence_table, course_table):
    # 1. Dapatkan hasil autocomplete langsung dari trie
    direct_results = trie.autocomplete(prefix)
    final_results = []

    for entry_id in direct_results:
        if entry_id not in final_results:
            final_results.append(entry_id)

        # 2. Cari padanan bahasa (INT <-> NAS) di equivalence_table jika ada
        equivalent_id = equivalence_table.get(entry_id)
        if equivalent_id:
            # Pastikan padanan tersebut ada di katalog course_table
            if course_table.contains(equivalent_id):
                if equivalent_id not in final_results:
                    final_results.append(equivalent_id)

    return final_results

def demo_trie():
    names, codes = Trie(), Trie()
    for entry in search_entries:
        names.insert(entry.name, entry.entry_id)
    for course in courses:
        codes.insert(course.course_code, course.course_code)
    assert set(names.autocomplete("ak")) == {"ACC13101", "ACC13102"}
    assert set(codes.autocomplete("ACC9")) == {"ACC93105", "ACC93106"}
    _, _, eq, catalog = build_indexes()
    result = bilingual_autocomplete(names, "Dasar", eq, catalog)
    assert set(result) == {"MGT12119", "MGT92119"}
    print("Saran dua bahasa:", result)

demo_or_todo("Trie", demo_trie)

Saran dua bahasa: ['MGT12119', 'MGT92119']


### 3.1 Penghubung Modul ke Data Asli
Sel berikut mengisi struktur buatan anggota memakai bahan pada bagian 0.4.4.
Aktifkan setelah Adit menyelesaikan `put/get` Hash Table dan Zilan menyelesaikan
`insert/autocomplete` Trie. Hubungan antarbagian:

1. Hash mata kuliah menerima kode MK → detail dan daftar kode kelas.
2. Hash kelas menerima kode kelas → detail jadwal dan catatan data.
3. Trie nama/kode menghasilkan ID berupa kode MK.
4. ID hasil dicari di Hash mata kuliah, kemudian daftar kelas dibaca di Hash kelas.

Fungsi inti Hash Table dan Trie tetap dikerjakan pemilik modul. Penghubung tidak
memakai `dict` sebagai pengganti Hash Table; dictionary hanya menampung detail
data dan kumpulan struktur yang dikembalikan.

In [15]:
def hubungkan_pencarian_feb(bahan):
    # Hindari key berulang menimpa detail kelas yang belum diperiksa.
    for nama in ["kelas", "mata_kuliah"]:
        keys = [key for key, _ in bahan[nama]]
        if any(key is None or not str(key).strip() for key in keys):
            raise ValueError(f"Kode pada bahan {nama} masih kosong; periksa laporan data.")
        if len(keys) != len(set(keys)):
            raise ValueError(f"Kode pada bahan {nama} berulang; pastikan sesi/duplikasi sebelum membangun indeks.")
    tabel_kelas, tabel_mk = HashTableChaining(), HashTableChaining()
    trie_nama, trie_kode = Trie(), Trie()
    for key, detail in bahan["kelas"]:
        tabel_kelas.put(key, detail)
    for key, detail in bahan["mata_kuliah"]:
        tabel_mk.put(key, detail)
    for entry in bahan["nama"]:
        trie_nama.insert(entry.name, entry.entry_id)
    for entry in bahan["kode"]:
        trie_kode.insert(entry.name, entry.entry_id)
    return {"kelas": tabel_kelas, "mata_kuliah": tabel_mk, "nama": trie_nama, "kode": trie_kode}

AKTIFKAN_PENCARIAN_FEB = True  # Ubah setelah implementasi anggota siap.
indeks_pencarian_feb = None
if AKTIFKAN_PENCARIAN_FEB:
    try:
        indeks_pencarian_feb = hubungkan_pencarian_feb(bahan_pencarian_feb)
    except NotImplementedError:
        print("Hash Table/Trie masih memiliki TODO. Lengkapi modul anggota terlebih dahulu.")
    else:
        print("Struktur pencarian terhubung dengan data FEB.")
else:
    print("Penghubung siap; belum diaktifkan karena implementasi Hash Table/Trie masih menunggu anggota.")

# Contoh pemakaian setelah indeks siap:
# detail_mk = indeks_pencarian_feb["mata_kuliah"].get("ACC93105")
# detail_kelas = [indeks_pencarian_feb["kelas"].get(k) for k in detail_mk["class_codes"]]
# kode_hasil = indeks_pencarian_feb["nama"].autocomplete("akun")
# Jika ada variasi nama, tampilkan setiap kode hasil sekali.

Struktur pencarian terhubung dengan data FEB.


## 4. Dua AVL — Pemakaian dan Kapasitas Ruangan
**Wildan + Asep · PPT 11, 13**

AVL pertama memakai key **`(utilization, room_id)`**, AVL kedua memakai
**`(capacity, room_id)`**. ID membedakan dua ruang bernilai sama. Tingkat pemakaian
terbesar ditampilkan dahulu untuk ranking; ini **tidak berarti** ruang tersibuk
harus dipakai untuk booking berikutnya. Booking mempertimbangkan kapasitas dan waktu.

Untuk 35 peserta, data ilustrasi memilih **F705 (40)**. Untuk 101 peserta hasilnya None.
Tiga ruang dengan pemakaian tertinggi: **F805 → F807 → F802**.
Ranking yang sama persentasenya mengikuti ID secara menurun pada penelusuran key terbalik.

Saat pemakaian berubah, hapus key lama dari indeks pemakaian lalu masukkan key baru
dan perbarui tabel ruang. Saat kapasitas berubah, lakukan hal yang sama pada indeks kapasitas.
Jangan mengubah key node di tempat karena dapat merusak urutan pohon.

Tinggi kosong=0, daun=1; balance factor=tinggi kiri−tinggi kanan, harus −1/0/1.
| Kasus | Urutan angka | Perbaikan |
|---|---|---|
| LL | 30, 20, 10 | Putar kanan |
| RR | 10, 20, 30 | Putar kiri |
| LR | 30, 10, 20 | Putar kiri anak, lalu kanan |
| RL | 10, 30, 20 | Putar kanan anak, lalu kiri |

Insert/delete/search/ceiling O(log n), satu rotasi O(1), tampilkan k hasil
O(log n + k) dengan penelusuran yang tepat; memori O(n) per indeks.
Mengurutkan semua ruang setiap query tidak memenuhi target top-k tersebut.
Dua indeks tetap O(n) total memori. Filter fasilitas/jenis dan jadwal tersedia
dapat menambah biaya; kapasitas cukup belum berarti ruang tersedia.

**Sudah tersedia:** rotasi, insert, traversal, pencarian kapasitas,
dua indeks, ranking, dan pemeriksaan tinggi/BST/keseimbangan.
**Tersedia:** delete dengan rebalance; pembaruan registry dan kedua indeks diuji pada bagian 6–7.


In [16]:
from typing import List, Optional

class AVLNode:
    def __init__(self, key: tuple, room: 'Room'):
        self.key = key  # (nilai_indeks, room_id)
        self.room = room
        self.left: Optional[AVLNode] = None
        self.right: Optional[AVLNode] = None
        self.height: int = 1

class AVLTree:
    def __init__(self, index="capacity"):
        if index not in ("capacity", "utilization"):
            raise ValueError("Indeks harus capacity atau utilization.")
        self.index = index
        self.rotations = 0
        self.root: Optional[AVLNode] = None

    def get_height(self, node: Optional[AVLNode]) -> int:
        return node.height if node else 0

    def get_balance(self, node: Optional[AVLNode]) -> int:
        if not node:
            return 0
        return self.get_height(node.left) - self.get_height(node.right)

    def update_height(self, node: AVLNode):
        node.height = 1 + max(self.get_height(node.left), self.get_height(node.right))

    def rotate_right(self, y: AVLNode) -> AVLNode:
        self.rotations += 1
        x = y.left
        assert x is not None
        T2 = x.right

        x.right = y
        y.left = T2

        self.update_height(y)
        self.update_height(x)
        return x

    def rotate_left(self, x: AVLNode) -> AVLNode:
        self.rotations += 1
        y = x.right
        assert y is not None
        T2 = y.left

        y.left = x
        x.right = T2

        self.update_height(x)
        self.update_height(y)
        return y

    def insert(self, key, room=None):
        # Gunakan insert(room) atau insert(key, room).
        if room is None:
            room = key
            key = (getattr(room, self.index), room.room_id)
        if key != (getattr(room, self.index), room.room_id):
            raise ValueError("Key tidak cocok dengan indeks dan data ruangan.")
        self.root = self._insert(self.root, key, room)

    def _insert(self, node: Optional[AVLNode], key: tuple, room: 'Room') -> AVLNode:
        if not node:
            return AVLNode(key, room)

        if key == node.key:
            node.room = room
            return node

        if key < node.key:
            node.left = self._insert(node.left, key, room)
        else:
            node.right = self._insert(node.right, key, room)

        self.update_height(node)
        balance = self.get_balance(node)

        # Left Left Case
        if balance > 1 and key < node.left.key:
            return self.rotate_right(node)

        # Right Right Case
        if balance < -1 and key > node.right.key:
            return self.rotate_left(node)

        # Left Right Case
        if balance > 1 and key > node.left.key:
            node.left = self.rotate_left(node.left)
            return self.rotate_right(node)

        # Right Left Case
        if balance < -1 and key < node.right.key:
            node.right = self.rotate_right(node.right)
            return self.rotate_left(node)

        return node

    def in_order_traversal(self) -> List['Room']:
        rooms_list = []
        self._in_order(self.root, rooms_list)
        return rooms_list

    def _in_order(self, node: Optional[AVLNode], rooms_list: list):
        if node:
            self._in_order(node.left, rooms_list)
            rooms_list.append(node.room)
            self._in_order(node.right, rooms_list)

    def find_suitable_room(self, participants: int) -> Optional['Room']:
        if self.index != "capacity":
            raise ValueError("Pencarian kapasitas harus memakai indeks capacity.")
        if type(participants) is not int or participants <= 0:
            raise ValueError("Jumlah peserta harus berupa bilangan bulat positif.")

        candidate: Optional['Room'] = None
        current = self.root

        while current:
            # Key format is (capacity, room_id)
            if current.room.capacity >= participants:
                candidate = current.room
                current = current.left  # Try to find a smaller but still sufficient capacity
            else:
                current = current.right # Capacity is too small, look for larger ones

        return candidate

    def check_balance(self) -> bool:
        return self._check_balance(self.root)

    def _check_balance(self, node: Optional[AVLNode]) -> bool:
        if not node:
            return True
        balance = self.get_balance(node)
        if abs(balance) > 1:
            return False
        return self._check_balance(node.left) and self._check_balance(node.right)



    # Operasi pencarian, ranking, dan validasi.
    def delete(self, key):
        """Hapus key O(log n); True bila ditemukan, False bila tidak ada."""
        def rebalance(node):
            self.update_height(node)
            balance = self.get_balance(node)
            if balance > 1:
                if self.get_balance(node.left) < 0:
                    node.left = self.rotate_left(node.left)
                return self.rotate_right(node)
            if balance < -1:
                if self.get_balance(node.right) > 0:
                    node.right = self.rotate_right(node.right)
                return self.rotate_left(node)
            return node

        def remove(node, target):
            if node is None:
                return None, False
            if target < node.key:
                node.left, found = remove(node.left, target)
            elif target > node.key:
                node.right, found = remove(node.right, target)
            else:
                found = True
                if node.left is None:
                    return node.right, True
                if node.right is None:
                    return node.left, True
                successor = node.right
                while successor.left is not None:
                    successor = successor.left
                node.key, node.room = successor.key, successor.room
                node.right, _ = remove(node.right, successor.key)
            return (rebalance(node), True) if found else (node, False)

        self.root, found = remove(self.root, key)
        return found

    def inorder(self):
        return self.in_order_traversal()

    def ceiling_capacity(self, participants):
        return self.find_suitable_room(participants)

    def top_k_utilization(self, k):
        if self.index != "utilization":
            raise ValueError("Ranking harus memakai indeks utilization.")
        if type(k) is not int or k < 0:
            raise ValueError("k harus bilangan bulat tidak negatif.")
        stack, result, node = [], [], self.root
        while (node or stack) and len(result) < k:
            while node:
                stack.append(node)
                node = node.right
            node = stack.pop()
            result.append(node.room)
            node = node.left
        return result

    def validate(self):
        def visit(node, lower=None, upper=None):
            if node is None:
                return 0
            assert lower is None or lower < node.key, "Urutan BST salah."
            assert upper is None or node.key < upper, "Urutan BST salah."
            assert node.key == (getattr(node.room, self.index), node.room.room_id)
            left_height = visit(node.left, lower, node.key)
            right_height = visit(node.right, node.key, upper)
            height = 1 + max(left_height, right_height)
            assert node.height == height, "Tinggi tersimpan salah."
            assert abs(left_height - right_height) <= 1, "Pohon tidak seimbang."
            return height
        visit(self.root)
        return True


def demo_avl():
    by_usage, by_capacity = AVLTree("utilization"), AVLTree("capacity")
    for room in rooms:
        by_usage.insert((room.utilization, room.room_id), room)
        by_capacity.insert((room.capacity, room.room_id), room)
    assert by_capacity.ceiling_capacity(35).room_id == "F705"
    assert by_capacity.ceiling_capacity(101) is None
    ranking = [r.room_id for r in by_usage.top_k_utilization(3)]
    assert ranking == ["F805", "F807", "F802"]
    assert by_usage.validate() and by_capacity.validate()
    print("Ruang untuk 35 peserta:", by_capacity.ceiling_capacity(35).room_id)
    print("Ranking pemakaian:", ranking)
    for name, sequence in [("LL", [30, 20, 10]), ("RR", [10, 20, 30]),
                           ("LR", [30, 10, 20]), ("RL", [10, 30, 20])]:
        tree = AVLTree()
        for value in sequence:
            tree.insert(Room(f"DEMO-{value}", value))
        assert tree.validate() and tree.root.key[0] == 20
        print(f"Rotasi {name}: seimbang, root kapasitas 20.")
    # Delete dan pembaruan kedua indeks diuji pada bagian 7.

demo_or_todo("AVL", demo_avl)


Ruang untuk 35 peserta: F705
Ranking pemakaian: ['F805', 'F807', 'F802']
Rotasi LL: seimbang, root kapasitas 20.
Rotasi RR: seimbang, root kapasitas 20.
Rotasi LR: seimbang, root kapasitas 20.
Rotasi RL: seimbang, root kapasitas 20.


## 5. DP — Pemilihan Jadwal Terbaik
**Rifqi + Abdur · PPT 14–15**

Pilih kegiatan yang tidak bertabrakan dan memiliki **total nilai terbesar**, untuk
**satu ruangan pada satu hari**. DP ditujukan terutama untuk ujian/event dengan waktu
beragam. Contoh slide 15 juga mencakup kuliah pengganti dan praktikum, sehingga contoh
notebook mengizinkan keempat kategori. Kuliah reguler yang memakai slot tetap
dialokasikan melalui layanan booking, bukan dipaksa memakai DP.

Urutkan kegiatan berdasarkan waktu selesai, lalu mulai, lalu ID untuk hasil konsisten.
`p(j)` adalah indeks kegiatan terakhir sebelumnya yang selesai <= mulai kegiatan j;
nilainya 0 jika tidak ada. Nomor pada penjelasan adalah 1..n, berbeda dari indeks list Python.

```text
DP[0] = 0
DP[j] = max(DP[j-1], nilai[j] + DP[p(j)])
```

Pilihan pertama melewati j, pilihan kedua mengambil j dan menggabungkannya dengan
hasil sebelumnya yang cocok. Cari p(j) dengan pencarian biner pada bagian sebelum j,
isi tabel secara iteratif, lalu telusuri mundur untuk daftar kegiatan terpilih.
Jika nilainya seri, pilih cabang lewati supaya rekonstruksi konsisten.

| j | Kegiatan | Waktu | Nilai | p(j) | DP[j] | Terpilih pada hasil akhir |
|---|---|---|---:|---:|---:|---|
| 1 | Seminar HIMA | 07.15–08.15 | 2 | 0 | 2 | Tidak |
| 2 | Kuliah pengganti | 08.15–09.05 | 5 | 1 | 7 | Tidak |
| 3 | UTS Akuntansi Manajerial | 07.15–09.15 | 10 | 0 | 10 | Ya |
| 4 | Sosialisasi KRS | 09.15–10.15 | 2 | 3 | 12 | Ya |
| 5 | UTS LAB | 10.15–11.45 | 7 | 4 | 19 | Ya |
| 6 | Kuliah pengganti | 11.00–12.40 | 5 | 4 | 19 | Tidak |

Hasil **J3 + J4 + J5 = 19**. Cara “ambil yang selesai paling cepat” menghasilkan
J1 + J2 + J4 + J5 = 16. Kolom terakhir merupakan pilihan akhir; baris pertama
sempat menjadi pilihan terbaik untuk satu kegiatan sebelum dipertimbangkan ulang.

### Definisi formal dan tiga implementasi DP

**State.** `OPT(j)` = nilai maksimum kombinasi kegiatan tidak bertabrakan dari
kegiatan `1..j` (sudah terurut waktu selesai). **Base case:** `OPT(0) = 0`.
**Recurrence:** `OPT(j) = max(OPT(j-1), nilai[j] + OPT(p(j)))` — lewati `j`,
atau ambil `j` plus solusi terbaik kegiatan yang selesai sebelum `j` mulai.

Tiga implementasi di bawah memakai definisi yang sama:

1. **Rekursi naif** — langsung dari recurrence tanpa cache; sub-masalah dihitung
   ulang sehingga kasus terburuk `O(2^n)` pemanggilan (input rantai, `p(j)=j-1`).
2. **Memoization (top-down)** — `OPT(j)` rekursif dengan cache dict manual;
   tiap state dihitung sekali, total `2n+1` pemanggilan untuk `n` kegiatan.
   Tie-break seri "lewati j" sama dengan tabulasi agar daftar ID identik.
3. **Tabulation (bottom-up)** — `optimize_schedule` mengisi `DP[1..n]` iteratif,
   lalu trace-back untuk daftar kegiatan terpilih.

**Analisis state × transisi:** `n` state (j = 1..n) × 2 transisi (lewati/ambil)
→ pengisian tabel O(n). Ditambah sorting O(n log n) dan p(j) via pencarian biner
O(n log n), total O(n log n) waktu dan O(n) ruang. Rekursi naif O(2^n) hanya
layak untuk `n` kecil; tabel pembanding pemanggilan/waktu ada di bagian 5.

**Optimasi ruang:** rolling array O(1) seperti interval scheduling unweighted
tidak berlaku di sini: `DP[j]` bergantung pada `DP[p(j)]` yang bisa jauh di
belakang, bukan hanya `DP[j-1]`. Nilai optimum saja tetap butuh array O(n), dan
rekonstruksi daftar kegiatan butuh tabel penuh, jadi ruang O(n) adalah batas
wajar implementasi ini. Memoisasi rekursif juga dibatasi kedalaman rekursi
Python (default ±1000); untuk `n` besar gunakan tabulasi yang iteratif.

Validasi: ID unik, satu ruang/hari, waktu valid, nilai >= 0. Input kosong menghasilkan
`(0, [])`. Kegiatan wajib memerlukan aturan tambahan; bobot tinggi tidak menjamin
setiap ujian dipilih. Konflik dosen/mahasiswa belum dimodelkan oleh DP ini.

Sort + pencarian p(j): O(n log n); isi tabel dan telusuri hasil: O(n); ruang O(n).
Memindai semua pendahulu mengganti biaya menjadi O(n²). Perbandingan semua kombinasi
hanya untuk n kecil, karena O(2^n · n).


In [17]:
def validate_candidates(candidates):
    if len({r.request_id for r in candidates}) != len(candidates):
        raise ValueError("ID kegiatan harus unik.")
    if len({(r.room_id, r.day) for r in candidates}) > 1:
        raise ValueError("DP hanya menerima satu ruangan dan satu hari.")
    if any(not (0 <= r.start < r.end <= 1440) or r.value < 0 for r in candidates):
        raise ValueError("Waktu atau nilai kegiatan tidak valid.")

def find_previous_compatible(ordered, j):
    """p(j) versi 1..n: posisi terakhir i < j dengan ordered[i-1].end <= ordered[j-1].start.
    Pencarian biner manual pada kegiatan 1..j-1 (sudah terurut berdasarkan end)."""
    target = ordered[j - 1].start
    low, high, answer = 1, j - 1, 0
    while low <= high:
        mid = (low + high) // 2
        if ordered[mid - 1].end <= target:
            answer = mid
            low = mid + 1
        else:
            high = mid - 1
    return answer

def optimize_schedule(candidates):
    validate_candidates(candidates)
    if not candidates:
        return 0, []

    # 1) Urutkan berdasarkan selesai, lalu mulai, lalu ID agar hasil konsisten.
    ordered = sorted(candidates, key=lambda r: (r.end, r.start, r.request_id))
    n = len(ordered)

    # 2) Hitung p(j) untuk j = 1..n dengan binary search. p[0] tidak dipakai.
    p = [0] * (n + 1)
    for j in range(1, n + 1):
        p[j] = find_previous_compatible(ordered, j)

    # 3) Isi tabel DP secara iteratif (bottom-up).
    dp = [0] * (n + 1)
    for j in range(1, n + 1):
        skip = dp[j - 1]
        take = ordered[j - 1].value + dp[p[j]]
        dp[j] = max(skip, take)

    # 4) Trace-back: jika seri, pilih cabang lewati.
    selected, j = [], n
    while j > 0:
        take = ordered[j - 1].value + dp[p[j]]
        if take > dp[j - 1]:
            selected.append(ordered[j - 1].request_id)
            j = p[j]
        else:
            j -= 1
    selected.reverse()
    return dp[n], selected

def optimize_schedule_memo(candidates):
    """Versi memoization (top-down): recurrence yang sama, ditulis rekursif.

    Definisi formal (sama untuk ketiga implementasi DP di notebook ini):
      OPT(0) = 0
      OPT(j) = max(OPT(j-1), nilai[j] + OPT(p(j)))
    Cache berupa dict {j: nilai} yang ditulis manual, bukan functools.lru_cache,
    supaya mekanisme DP-nya terlihat. Tie-break seri tetap "lewati j" sehingga
    daftar ID terpilih identik dengan optimize_schedule (tabulation).
    Return (nilai_optimum, daftar_id, jumlah_pemanggilan_OPT).
    """
    validate_candidates(candidates)
    if not candidates:
        return 0, [], 0
    ordered = sorted(candidates, key=lambda r: (r.end, r.start, r.request_id))
    n = len(ordered)
    p = [0] * (n + 1)
    for j in range(1, n + 1):
        p[j] = find_previous_compatible(ordered, j)
    cache = {0: 0}
    calls = 0

    def opt(j):
        nonlocal calls
        calls += 1
        if j not in cache:
            skip = opt(j - 1)
            take = ordered[j - 1].value + opt(p[j])
            cache[j] = skip if skip >= take else take  # seri -> lewati j.
        return cache[j]

    try:
        best = opt(n)  # Kedalaman rekursi ~n+1; untuk n besar gunakan tabulasi.
    except RecursionError:
        raise RecursionError("Batas rekursi Python terlampaui; gunakan tabulasi untuk n besar.")
    selected, j = [], n  # Trace-back memakai cache yang sudah terisi penuh.
    while j > 0:
        take = ordered[j - 1].value + cache[p[j]]
        if take > cache[j - 1]:
            selected.append(ordered[j - 1].request_id)
            j = p[j]
        else:
            j -= 1
    selected.reverse()
    return best, selected, calls

def explain_dp_table(candidates):
    """Cetak tabel j, kegiatan, waktu, nilai, p(j), DP[j] seperti markdown slide 15."""
    validate_candidates(candidates)
    ordered = sorted(candidates, key=lambda r: (r.end, r.start, r.request_id))
    n = len(ordered)
    p = [0] + [find_previous_compatible(ordered, j) for j in range(1, n + 1)]
    dp = [0] * (n + 1)
    for j in range(1, n + 1):
        dp[j] = max(dp[j - 1], ordered[j - 1].value + dp[p[j]])
    _, chosen = optimize_schedule(candidates)
    fmt = lambda m: f"{m // 60:02d}.{m % 60:02d}"
    print(f"{'j':>2} | {'ID':<4} | {'Kegiatan':<28} | {'Waktu':<11} | {'Nilai':>5} | {'p(j)':>4} | {'DP[j]':>5} | Terpilih")
    for j in range(1, n + 1):
        r = ordered[j - 1]
        print(f"{j:>2} | {r.request_id:<4} | {r.title[:28]:<28} | {fmt(r.start)}–{fmt(r.end)} | "
              f"{r.value:>5} | {p[j]:>4} | {dp[j]:>5} | {'Ya' if r.request_id in chosen else 'Tidak'}")

def demo_dp():
    total, selected = optimize_schedule(candidate_schedules)
    assert total == 19 and set(selected) == {"J3", "J4", "J5"}
    assert optimize_schedule([]) == (0, [])
    memo_total, memo_selected, memo_calls = optimize_schedule_memo(candidate_schedules)
    assert (memo_total, memo_selected) == (total, selected), "Memoisasi harus identik dengan tabulasi."
    explain_dp_table(candidate_schedules)
    print("Jadwal terbaik:", selected, "nilai:", total,
          "| memoisasi:", memo_calls, "pemanggilan OPT")

demo_or_todo("DP", demo_dp)


 j | ID   | Kegiatan                     | Waktu       | Nilai | p(j) | DP[j] | Terpilih
 1 | J1   | Seminar HIMA                 | 07.15–08.15 |     2 |    0 |     2 | Tidak
 2 | J2   | Kelas pengganti 1 SKS        | 08.15–09.05 |     5 |    1 |     7 | Tidak
 3 | J3   | UTS Akuntansi Manajerial     | 07.15–09.15 |    10 |    0 |    10 | Ya
 4 | J4   | Sosialisasi KRS              | 09.15–10.15 |     2 |    3 |    12 | Ya
 5 | J5   | UTS LAB Pengantar Akuntansi  | 10.15–11.45 |     7 |    4 |    19 | Ya
 6 | J6   | Kelas pengganti 2 SKS        | 11.00–12.40 |     5 |    4 |    19 | Tidak
Jadwal terbaik: ['J3', 'J4', 'J5'] nilai: 19


### 5.1 Pembanding yang sudah bisa dijalankan
Dua fungsi berikut adalah **alat pemeriksa**, bukan implementasi DP. Pemeriksaan semua
kombinasi dibatasi maksimal 15 kegiatan supaya tidak berjalan terlalu lama.
Pemilihan berdasarkan waktu selesai menunjukkan mengapa jumlah kegiatan dan total
nilai adalah dua tujuan yang berbeda. Kasus acak juga perlu diperiksa setelah DP diisi.


In [18]:
def no_overlap(selected):
    ordered = sorted(selected, key=lambda r: (r.start, r.end, r.request_id))
    return all(a.end <= b.start for a, b in zip(ordered, ordered[1:]))

def brute_force_reference(candidates):
    validate_candidates(candidates)
    if len(candidates) > 15:
        raise ValueError("Pembanding dibatasi 15 kegiatan.")
    candidates = sorted(candidates, key=lambda r: (r.start, r.end, r.request_id))
    best_value, best_ids = 0, []
    for size in range(len(candidates) + 1):
        for chosen in combinations(candidates, size):
            if all(a.end <= b.start for a, b in zip(chosen, chosen[1:])):
                value = sum(r.value for r in chosen)
                if value > best_value:
                    best_value = value
                    best_ids = [r.request_id for r in chosen]
    return best_value, best_ids

def earliest_finish_reference(candidates):
    validate_candidates(candidates)
    selected, last_end = [], -1
    for request in sorted(candidates, key=lambda r: (r.end, r.start, r.request_id)):
        if request.start >= last_end:
            selected.append(request)
            last_end = request.end
    return sum(r.value for r in selected), [r.request_id for r in selected]

brute_result = brute_force_reference(candidate_schedules)
greedy_result = earliest_finish_reference(candidate_schedules)
assert brute_result == (19, ["J3", "J4", "J5"])
assert greedy_result == (16, ["J1", "J2", "J4", "J5"])
print("Semua kombinasi:", brute_result)
print("Selesai paling cepat:", greedy_result)

def verify_dp_random_cases(case_count=100, seed=SEED):
    rng = Random(seed)
    for case in range(case_count):
        sample = generate_requests(rng.randrange(0, 16), seed=rng.randrange(1_000_000))
        expected, _ = brute_force_reference(sample)
        actual, ids = optimize_schedule(sample)
        selected = [r for r in sample if r.request_id in set(ids)]
        assert len(ids) == len(set(ids)) == len(selected), f"ID salah pada kasus {case}"
        assert no_overlap(selected) and sum(r.value for r in selected) == actual
        assert actual == expected, f"Nilai berbeda pada kasus {case}"
    return f"{case_count} kasus acak DP sesuai pembanding."

def naive_recursive(candidates):
    """Rekursi naif: recurrence yang sama tanpa cache, pembanding O(2^n).

    Setiap OPT(j) memanggil OPT(j-1) dan OPT(p(j)) dari nol, sehingga sub-masalah
    dihitung ulang dan jumlah pemanggilan tumbuh eksponensial pada kasus terburuk
    (input rantai kompatibel, p(j)=j-1: tepat 2^(n+1)-1 pemanggilan).
    Return (nilai_optimum, jumlah_pemanggilan_OPT); tanpa rekonstruksi daftar ID
    karena tujuannya mengukur biaya, bukan menghasilkan jadwal.
    """
    validate_candidates(candidates)
    if len(candidates) > 26:
        raise ValueError("Rekursi naif dibatasi 26 kegiatan karena eksponensial.")
    if not candidates:
        return 0, 0
    ordered = sorted(candidates, key=lambda r: (r.end, r.start, r.request_id))
    n = len(ordered)
    p = [0] * (n + 1)
    for j in range(1, n + 1):
        p[j] = find_previous_compatible(ordered, j)
    calls = 0

    def opt(j):
        nonlocal calls
        calls += 1
        if j == 0:
            return 0
        skip = opt(j - 1)
        take = ordered[j - 1].value + opt(p[j])
        return skip if skip >= take else take

    return opt(n), calls

def chain_requests(n, seed=SEED):
    """Rantai kegiatan pendek yang saling kompatibel: p(j) = j-1 untuk semua j.

    Kasus terburuk rekursi naif (kedua cabang recurrence menyelam penuh,
    2^(n+1)-1 pemanggilan) sekaligus kasus normal bagi memo/tabulasi (O(n)).
    Dipakai bersama generate_requests sebagai dua jenis input pembanding yang
    sama untuk ketiga metode. Seed hanya menggeser jam mulai; rantai tetap
    kompatibel apa pun seed-nya.
    """
    rng = Random(seed)
    first = 420 + rng.randrange(0, 30)
    return [ScheduleRequest(f"CHAIN-{i:02d}", f"Kegiatan rantai {i}", "Event", 10, "F802",
                            DEMO_DAY, first + 30 * i, first + 30 * i + 30, 5, i)
            for i in range(n)]

def compare_dp_methods(sizes=(5, 10, 15, 20, 25), chain_sizes=(5, 10, 15, 20), seed=SEED):
    """Tabel naif vs memoization vs tabulation pada input yang sama.

    Dua jenis input: "padat" (generate_requests; tabrakan banyak, p(j) kecil)
    dan "rantai" (chain_requests; p(j)=j-1, kasus terburuk rekursi naif).
    Kolom pemanggilan menghitung masuknya OPT() untuk naif/memo; tabulation
    menghitung sel DP yang diisi, yaitu n. Return daftar dict baris dan
    mencetak tabel; nilai optimum ketiga metode dan ID memo vs tabulasi
    di-assert sama.
    """
    rows = []
    for label, batch_sizes, make in (("padat", sizes, generate_requests),
                                     ("rantai", chain_sizes, chain_requests)):
        for n in batch_sizes:
            batch = make(n, seed)
            started = perf_counter()
            naive_value, naive_calls = naive_recursive(batch)
            naive_seconds = perf_counter() - started
            started = perf_counter()
            memo_value, memo_ids, memo_calls = optimize_schedule_memo(batch)
            memo_seconds = perf_counter() - started
            started = perf_counter()
            tab_value, tab_ids = optimize_schedule(batch)
            tab_seconds = perf_counter() - started
            assert naive_value == memo_value == tab_value, f"Nilai berbeda pada {label} n={n}."
            assert memo_ids == tab_ids, f"ID memo != tabulasi pada {label} n={n}."
            rows.append({"input": label, "n": n, "nilai": tab_value,
                         "naif_pemanggilan": naive_calls, "naif_detik": round(naive_seconds, 6),
                         "memo_pemanggilan": memo_calls, "memo_detik": round(memo_seconds, 6),
                         "tab_sel": n, "tab_detik": round(tab_seconds, 6)})
    print("naif vs memo vs tabulasi (pemanggilan OPT; tabulasi = sel yang diisi)")
    print(f"{'input':>6} | {'n':>3} | {'naif':>12} {'detik':>9} | {'memo':>8} {'detik':>9} | {'tab':>4} {'detik':>9}")
    for row in rows:
        print(f"{row['input']:>6} | {row['n']:>3} | {row['naif_pemanggilan']:>12} {row['naif_detik']:>9.6f} | "
              f"{row['memo_pemanggilan']:>8} {row['memo_detik']:>9.6f} | "
              f"{row['tab_sel']:>4} {row['tab_detik']:>9.6f}")
    return rows

# Jalankan setelah implementasi DP selesai:
print(verify_dp_random_cases())

# Tabel Level 2 (S7): jumlah pemanggilan dan waktu naif vs memo vs tabulasi.
method_rows = compare_dp_methods()


Semua kombinasi: (19, ['J3', 'J4', 'J5'])
Selesai paling cepat: (16, ['J1', 'J2', 'J4', 'J5'])
100 kasus acak DP sesuai pembanding.


## 6. Integrasi — Zacky

**Terimplementasi:** penggabungan kelas, pembaruan registry secara transaksional,
antrean PQ, DP per ruang/hari, percobaan ruang alternatif, dan simulasi Excel.
Operasi delete AVL bagian 4 dilengkapi karena diperlukan untuk menghapus key lama.

### Alur dan kontrak

1. `merge_equivalent_classes(records, equivalence_table, room_table)` menerima
   pasangan **eksplisit dua arah yang telah diverifikasi caller**. Hanya kelas dengan
   hari, jam, ruang, dosen yang terisi, dan kategori sama yang digabung. Kedua kode
   pasangan harus hadir; kelas satu kode saja tidak otomatis digabung. Kapasitas
   diperiksa setelah peserta dijumlahkan. Hasil `MergedClassRecord` menyimpan
   `source_ids` dan `course_codes`; ID asal berulang ditolak. Input tidak diubah.
2. `make_room_registry(rooms)` membangun Hash ruang serta AVL kapasitas/pemakaian.
   `update_room_usage(table, usage_avl, room_id, slots, capacity_avl=None)` memvalidasi
   0..15, menghapus key lama, mengganti Room, dan memasukkan key/payload baru.
   Berikan indeks kapasitas sebagai argumen kelima agar **kedua indeks** ikut baru.
   Staging salinan dan validasi selesai sebelum commit; kegagalan uji tidak
   meninggalkan registry setengah berubah. Biaya transaksi mencakup salinan O(R),
   berbeda dari operasi AVL murni O(log R).
3. `schedule_across_rooms(requests, table, capacity_avl, usage_avl, ...)` membangun
   PQ terpisah per tanggal. Prioritas lalu jam lalu FIFO menentukan urutan masukan;
   DP memilih kombinasi bernilai terbesar pada batch **satu ruang/hari**.
   Ruang yang diminta dicoba lebih dahulu, lalu calon ruang menurut kapasitas/ID.
4. Kapasitas, interval booking yang sudah diterima, dan kebutuhan jenis ruang
   eksplisit diperiksa. `required_room_types` opsional; tanpa metadata tersebut,
   fasilitas tidak diklaim sudah diperiksa. `allow_alternative_rooms=False`
   membatasi percobaan pada ruang awal/calon pertama.
5. Permintaan yang tidak terpilih boleh mencoba ruang lain. Setiap pasangan
   `(request_id, room_id)` dicoba paling banyak sekali; maksimal N×R percobaan.
   Booking yang telah diterima tidak digeser pada putaran berikutnya. Hasil memuat
   `bookings`, `rejected` beserta alasan, `attempts`, `tried_rooms`, `priority_order`,
   `total_value`, dan `occupied_slots`. Setiap input berakhir diterima atau ditolak.

### Pemakaian ruang dan kondisi awal

Pemakaian dihitung dari himpunan `(tanggal, indeks slot reguler)` yang tersentuh,
bukan `used_slots += jumlah_kegiatan`. Dua kegiatan bersebelahan pada slot reguler
yang sama menambah **satu** slot unik. Akhir pekan tidak masuk penyebut 15 slot
Senin–Jumat. Input dibatasi satu minggu ISO; minggu berbeda disimulasikan terpisah.

Untuk ruang yang sudah terpakai, berikan `existing_bookings` atau `occupied_slots`
yang sesuai `used_slots`. Angka total saja tidak cukup untuk mengetahui waktu kosong.
Slot terpakai tanpa interval rinci diblokir secara konservatif. Booking awal yang
saling bentrok ditolak sebelum registry diubah.

### Data FEB dan contoh pengujian

`simulate_feb_allocation(data_bersih, bahan_pencarian_feb)` memakai kelas dari Excel.
**Simulasi dimulai dari ruang kosong**, bukan mengubah atau memvalidasi seluruh
jadwal resmi. Kapasitas memakai `kapasitas_asumsi`; jenis/fasilitas ruang Unknown.
Baris online, durasi/ruang yang perlu pemeriksaan, dan input tidak valid dilaporkan
terpisah beserta baris Excel. Semua kandidat adalah ujian prioritas 1/bobot 10.
Pasangan INT–NAS pada Excel belum dikonfirmasi resmi sehingga tidak digabung otomatis.

Fixture J1–J6 tetap khusus pengujian: nilai DP 19 dan 2 slot reguler unik.
Fixture pasangan 15+20=35 memeriksa aturan penggabungan, bukan fakta kapasitas FEB.

**Batas:** strategi beberapa ruang tidak menjamin optimum global. DP tidak
menjadikan prioritas Heap sebagai batas wajib, belum memodelkan konflik
dosen/mahasiswa atau kegiatan wajib. Semua alokasi berada di memori simulasi;
fungsi tidak menulis perubahan ke Excel.


In [19]:
from copy import deepcopy
from datetime import date
import json


@dataclass(frozen=True)
class MergedClassRecord(ClassRecord):
    source_ids: tuple = ()
    course_codes: tuple = ()


def merge_equivalent_classes(records, equivalence_table, room_table):
    """Gabung hanya pasangan eksplisit dua arah yang telah diverifikasi caller.

    Kelas dengan dosen kosong tidak digabung. ID asal disimpan agar hasil dapat
    ditelusuri, tidak dihitung dua kali, dan aman dipanggil ulang (idempoten).
    Tidak mengubah records, tabel pasangan, atau tabel ruang.
    """
    groups, seen = HashTableChaining(), HashTableChaining()
    for record in records:
        if not isinstance(record, ClassRecord) or not record.class_code:
            raise ValueError("Record/kode kelas tidak valid.")
        if type(record.participants) is not int or record.participants <= 0:
            raise ValueError("Peserta harus bilangan bulat positif.")
        if not record.day or not 0 <= record.start < record.end <= 1440:
            raise ValueError("Tanggal atau interval kelas tidak valid.")
        date.fromisoformat(record.day)
        if record.activity_type not in PRIORITY:
            raise ValueError("Kategori tidak dikenal.")
        room = room_table.get(record.room_id)
        if room is None or record.participants > room.capacity:
            raise ValueError(f"Ruang tidak tersedia/kapasitas kurang: {record.class_code}")
        origins = getattr(record, "source_ids", ()) or (record.class_code,)
        for identifier in origins:
            if not isinstance(identifier, str) or not identifier or seen.contains(identifier):
                raise ValueError("ID asal kosong/berulang; peserta berisiko dihitung ganda.")
            seen.put(identifier, True)
        codes = getattr(record, "course_codes", ()) or (record.course_code,)
        partner = equivalence_table.get(record.course_code)
        pair = tuple(sorted((record.course_code, partner))) if (
            isinstance(partner, str) and partner != record.course_code and
            equivalence_table.get(partner) == record.course_code) else (record.course_code,)
        identity = [pair, record.day, record.start, record.end, record.room_id,
                    record.lecturer_id, record.activity_type]
        # Tanpa dosen atau pasangan eksplisit, setiap record tetap terpisah.
        if len(pair) != 2 or not record.lecturer_id:
            identity.append(record.class_code)
        key = json.dumps(identity, ensure_ascii=False)
        batch = groups.get(key)
        if batch is None:
            batch = []
            groups.put(key, batch)
        batch.append((record, tuple(origins), tuple(codes), pair))
    result = []
    for _, group in groups.items():
        all_codes = tuple(sorted({code for _, _, codes, _ in group for code in codes}))
        pair = group[0][3]
        batches = [group] if len(pair) == 2 and set(pair) <= set(all_codes) else [[entry] for entry in group]
        for batch in batches:
            first = batch[0][0]
            origins = tuple(sorted(origin for _, ids, _, _ in batch for origin in ids))
            codes = tuple(sorted({code for _, _, codes, _ in batch for code in codes}))
            people = sum(record.participants for record, _, _, _ in batch)
            if people > room_table.get(first.room_id).capacity:
                raise ValueError(f"Gabungan {origins} melebihi kapasitas ruang.")
            identifier = first.class_code if len(batch) == 1 else "MERGE:" + json.dumps(origins)
            result.append(MergedClassRecord(identifier, first.course_code, people,
                first.lecturer_id, first.room_id, first.day, first.start, first.end,
                first.activity_type, origins, codes))
    if len({r.class_code for r in result}) != len(result):
        raise ValueError("Kode hasil gabungan bertabrakan dengan kode kelas lain.")
    return result


def validate_room_registry(room_table, usage_avl, capacity_avl=None):
    if usage_avl.index != "utilization" or (capacity_avl is not None and capacity_avl.index != "capacity"):
        raise ValueError("Jenis indeks AVL tidak sesuai.")
    usage_avl.validate()
    if capacity_avl is not None:
        capacity_avl.validate()
    for key, room in room_table.items():
        if key != room.room_id or type(room.capacity) is not int or room.capacity <= 0:
            raise ValueError("Identitas/kapasitas ruang tidak valid.")
        if type(room.used_slots) is not int or not 0 <= room.used_slots <= WEEKLY_SLOTS:
            raise ValueError("Pemakaian ruang harus 0..15 slot.")
    for tree in [usage_avl] + ([capacity_avl] if capacity_avl is not None else []):
        entries = tree.inorder()
        if len(entries) != len(room_table):
            raise ValueError("Hash dan payload kedua indeks AVL tidak konsisten.")
        visited = HashTableChaining()
        for room in entries:
            if visited.contains(room.room_id) or room_table.get(room.room_id) != room:
                raise ValueError("Hash dan payload kedua indeks AVL tidak konsisten.")
            visited.put(room.room_id, True)
    return True


def stage_room_updates(room_table, usage_avl, capacity_avl, updates):
    """Salin registry untuk transaksi: validasi/gagal tidak merusak state asal.

    Biaya salinan/validasi O(R + bucket), berbeda dari delete/insert AVL O(log R).
    """
    validate_room_registry(room_table, usage_avl, capacity_avl)
    table, usage, capacity = deepcopy((room_table, usage_avl, capacity_avl))
    for room_id, used_slots in updates:
        old = table.get(room_id)
        if old is None:
            raise KeyError(f"Ruang tidak ditemukan: {room_id}")
        if type(used_slots) is not int or not 0 <= used_slots <= WEEKLY_SLOTS:
            raise ValueError("Pemakaian ruang harus bilangan bulat 0..15.")
        new = replace(old, used_slots=used_slots)
        if not usage.delete((old.utilization, room_id)):
            raise ValueError("Key pemakaian lama tidak ditemukan.")
        usage.insert(new)
        if capacity is not None:
            if not capacity.delete((old.capacity, room_id)):
                raise ValueError("Key kapasitas lama tidak ditemukan.")
            capacity.insert(new)
        table.put(room_id, new)
    validate_room_registry(table, usage, capacity)
    return table, usage, capacity


def commit_room_registry(room_table, usage_avl, capacity_avl, staged):
    table, usage, capacity = staged
    room_table.buckets, room_table.size = table.buckets, table.size
    room_table.probes = table.probes
    usage_avl.root, usage_avl.rotations = usage.root, usage.rotations
    if capacity_avl is not None:
        capacity_avl.root, capacity_avl.rotations = capacity.root, capacity.rotations


def update_room_usage(room_table, usage_avl, room_id, new_used_slots, capacity_avl=None):
    """Kompatibel 4 argumen; berikan capacity_avl agar payload kedua indeks ikut baru."""
    staged = stage_room_updates(room_table, usage_avl, capacity_avl, [(room_id, new_used_slots)])
    commit_room_registry(room_table, usage_avl, capacity_avl, staged)
    return room_table.get(room_id)


def touched_regular_slots(request):
    day = date.fromisoformat(request.day)
    if day.weekday() >= 5:
        return set()
    return {(request.day, index) for index, (start, end) in enumerate(REGULAR_SLOTS)
            if request.start < end and start < request.end}


def schedule_across_rooms(requests, room_table, capacity_avl, usage_avl,
                          existing_bookings=(), occupied_slots=None,
                          allow_alternative_rooms=True, required_room_types=None):
    """Simulasi alokasi satu minggu: PQ -> batch DP per ruang/hari -> retry.

    Booking yang diterima tidak digeser pada putaran berikutnya. Tiap pasangan
    (request_id, room_id) dicoba paling banyak sekali. Bukan optimum global;
    prioritas mengatur urutan antrean, DP memaksimalkan value dalam batch.
    Fasilitas hanya diperiksa bila required_room_types diberikan secara eksplisit.
    """
    validate_room_registry(room_table, usage_avl, capacity_avl)
    requests, existing = list(requests), list(existing_bookings)
    seen = HashTableChaining()
    weeks = set()
    for request in requests + existing:
        # Validasi kontrak Heap tanpa mengubah registry.
        MinHeapPriorityQueue()._check_request(request)
        if seen.contains(request.request_id):
            raise ValueError("ID permintaan/booking berulang.")
        seen.put(request.request_id, True)
        if type(request.participants) is not int or request.participants <= 0 or request.value < 0:
            raise ValueError("Peserta/nilai permintaan tidak valid.")
        day = date.fromisoformat(request.day)
        weeks.add(tuple(day.isocalendar()[:2]))
    if len(weeks) > 1:
        raise ValueError("Pemakaian /15 memerlukan input dari satu minggu ISO.")
    occupied, known, calendar = HashTableChaining(), HashTableChaining(), HashTableChaining()
    for room_id, room in room_table.items():
        occupied.put(room_id, set())
        known.put(room_id, set())
    for booking in existing:
        room = room_table.get(booking.room_id)
        if room is None or booking.participants > room.capacity:
            raise ValueError("Booking awal memakai ruang/kapasitas tidak valid.")
        key = json.dumps((booking.room_id, booking.day))
        batch = calendar.get(key) or []
        if any(booking.start < other.end and other.start < booking.end for other in batch):
            raise ValueError("Booking awal saling bertabrakan.")
        calendar.put(key, batch + [booking])
        known.get(booking.room_id).update(touched_regular_slots(booking))
    for room_id, room in room_table.items():
        supplied = set(occupied_slots.get(room_id, ())) if occupied_slots is not None else set(known.get(room_id))
        if not known.get(room_id) <= supplied or len(supplied) != room.used_slots:
            raise ValueError("Butuh posisi slot awal yang sesuai used_slots, bukan hanya totalnya.")
        for day_text, index in supplied:
            day = date.fromisoformat(day_text)
            if day.weekday() >= 5 or type(index) is not int or not 0 <= index < len(REGULAR_SLOTS):
                raise ValueError("Posisi slot awal tidak valid.")
            if weeks and tuple(day.isocalendar()[:2]) not in weeks:
                raise ValueError("Slot awal berasal dari minggu berbeda.")
        occupied.put(room_id, supplied)

    queues = HashTableChaining()
    for request in requests:
        queue = queues.get(request.day)
        if queue is None:
            queue = MinHeapPriorityQueue()
            queues.put(request.day, queue)
        queue.push(request)
    pending = []
    for day, queue in sorted(queues.items()):
        while queue:
            pending.append(queue.pop())
    priority_order = [r.request_id for r in pending]
    room_order = capacity_avl.inorder()
    tried, reasons = HashTableChaining(), HashTableChaining()
    for request in requests:
        tried.put(request.request_id, [])
    attempts, bookings, rejected = [], [], []
    while pending:
        batches = HashTableChaining()
        retry = []
        for request in pending:
            preferred = room_table.get(request.room_id)
            candidates = ([preferred] if preferred else []) + [r for r in room_order if r.room_id != request.room_id]
            if not allow_alternative_rooms:
                candidates = candidates[:1] if preferred or not request.room_id else []
            selected_room = None
            for room in candidates:
                history = tried.get(request.request_id)
                if room.room_id in history:
                    continue
                history.append(room.room_id)
                reason = None
                if room.capacity < request.participants:
                    reason = "Kapasitas kurang"
                required = required_room_types.get(request.request_id) if required_room_types else None
                if reason is None and required and room.room_type != required:
                    reason = "Jenis/fasilitas ruang tidak sesuai atau tidak diketahui"
                batch_key = json.dumps((room.room_id, request.day))
                if reason is None and any(request.start < other.end and other.start < request.end
                                          for other in calendar.get(batch_key) or []):
                    reason = "Bertabrakan dengan booking yang sudah diterima"
                if reason is None and touched_regular_slots(request) & (occupied.get(room.room_id) - known.get(room.room_id)):
                    reason = "Slot awal terpakai, interval rinci belum diketahui"
                if reason:
                    attempts.append({"request_id": request.request_id, "room_id": room.room_id,
                                     "status": "DITOLAK", "reason": reason})
                    reasons.put(request.request_id, reason)
                    continue
                selected_room = room
                batch = batches.get(batch_key) or []
                batch.append(replace(request, room_id=room.room_id))
                batches.put(batch_key, batch)
                break
            if selected_room is None:
                rejected.append({"request": request, "reason": reasons.get(request.request_id) or "Tidak ada ruang yang dapat dicoba"})
        for batch_key, batch in batches.items():
            _, ids = optimize_schedule(batch)
            chosen = set(ids)
            for request in batch:
                accepted = request.request_id in chosen
                attempts.append({"request_id": request.request_id, "room_id": request.room_id,
                    "status": "DITERIMA" if accepted else "DITOLAK",
                    "reason": "Terpilih DP batch" if accepted else "Tidak terpilih DP batch"})
                if accepted:
                    bookings.append(request)
                    calendar.put(batch_key, (calendar.get(batch_key) or []) + [request])
                    slots = touched_regular_slots(request)
                    occupied.get(request.room_id).update(slots)
                    known.get(request.room_id).update(slots)
                else:
                    reasons.put(request.request_id, "Tidak terpilih DP batch")
                    retry.append(request)
        pending = retry
    updates = [(room_id, len(slots)) for room_id, slots in occupied.items()]
    staged = stage_room_updates(room_table, usage_avl, capacity_avl, updates)
    commit_room_registry(room_table, usage_avl, capacity_avl, staged)
    return {"bookings": bookings, "rejected": rejected, "attempts": attempts,
            "tried_rooms": tried, "priority_order": priority_order,
            "total_value": sum(r.value for r in bookings), "occupied_slots": occupied,
            "limits": ["Simulasi satu minggu; bukan perubahan jadwal Excel.",
                       "DP optimal per batch ruang/hari, bukan optimum global beberapa ruang.",
                       "Konflik dosen/mahasiswa belum dimodelkan; fasilitas memerlukan metadata eksplisit."]}


def make_room_registry(room_list):
    table, capacity, usage = HashTableChaining(), AVLTree("capacity"), AVLTree("utilization")
    for room in room_list:
        if table.contains(room.room_id):
            raise ValueError("ID ruang berulang.")
        table.put(room.room_id, room)
        capacity.insert(room)
        usage.insert(room)
    validate_room_registry(table, usage, capacity)
    return table, capacity, usage


def demo_integration():
    table, classes, pairs, catalog = build_indexes()
    names = Trie()
    for course in courses:
        names.insert(course.name, course.course_code)
    assert set(bilingual_autocomplete(names, "Dasar", pairs, catalog)) == {"MGT12119", "MGT92119"}
    merged = merge_equivalent_classes([record for _, record in classes.items()], pairs, table)
    assert len(merged) == 1 and merged[0].participants == 35
    # Registry kosong untuk simulasi J1–J6; bukan reset pemakaian kampus nyata.
    registry = make_room_registry([Room("F802", 60)])
    report = schedule_across_rooms(requests, *registry, allow_alternative_rooms=False)
    assert report["total_value"] == 19
    assert {r.request_id for r in report["bookings"]} == {"J3", "J4", "J5"}
    assert registry[0].get("F802").used_slots == 2
    print("Integrasi fixture: pasangan 15+20=35, DP=19, slot unik F802=2.")
    return report


def simulate_feb_allocation(clean_data, ingredients):
    """Excel sebagai permintaan simulasi, bukan penulisan ulang jadwal resmi.

    Durasi/ruang bermasalah dan online dilaporkan terpisah. Kapasitas asumsi dan
    jenis ruang tidak diketahui diberi label; pasangan belum resmi tidak digabung.
    """
    room_list = []
    invalid_rooms = []
    for _, row in clean_data["ruang"].iterrows():
        if pd.isna(row["ruang"]) or pd.isna(row["kapasitas_asumsi"]) or row["kapasitas_asumsi"] <= 0:
            invalid_rooms.append(int(row["baris_excel"]))
        else:
            room_list.append(Room(str(row["ruang"]), int(row["kapasitas_asumsi"]), "Unknown", 0))
    candidates, excluded, provenance = [], [], HashTableChaining()
    for code, detail in ingredients["kelas"]:
        reason = None
        if str(detail["exam_mode"]).casefold() != "onsite": reason = "Online: tidak memakai ruang fisik"
        elif detail["needs_review"]: reason = "Durasi/ruang perlu pemeriksaan"
        elif not detail["day"] or detail["start"] is None or detail["end"] is None: reason = "Tanggal/jam belum valid"
        elif not 0 <= detail["start"] < detail["end"] <= 1440: reason = "Interval tidak valid"
        elif detail["participants"] is None or detail["participants"] <= 0: reason = "Peserta tidak valid"
        if reason:
            excluded.append({"class_code": code, "excel_row": detail["source"]["excel_row"], "reason": reason})
            continue
        candidates.append(ScheduleRequest(code, detail["name"], "Examination", int(detail["participants"]),
            detail["room_id"] or "", detail["day"], int(detail["start"]), int(detail["end"]), VALUE["Examination"]))
        provenance.put(code, detail["source"])
    by_week = HashTableChaining()
    for request in candidates:
        key = str(tuple(date.fromisoformat(request.day).isocalendar()[:2]))
        by_week.put(key, (by_week.get(key) or []) + [request])
    reports = []
    for week, batch in sorted(by_week.items()):
        registry = make_room_registry(room_list)
        report = schedule_across_rooms(batch, *registry)
        report["week"] = week
        report["registry"] = registry
        reports.append(report)
    return {"weeks": reports, "excluded": excluded, "invalid_room_rows": invalid_rooms,
            "provenance": provenance, "pairs_merged": 0,
            "assumptions": "Simulasi mulai ruang kosong; kapasitas asumsi Excel, fasilitas belum diketahui, pasangan belum resmi tidak digabung."}


integration_demo = demo_integration()
feb_allocation_demo = simulate_feb_allocation(data_bersih, bahan_pencarian_feb)
print("SIMULASI EXCEL:", feb_allocation_demo["assumptions"])
for report in feb_allocation_demo["weeks"]:
    print("Minggu", report["week"], "diterima", len(report["bookings"]),
          "ditolak", len(report["rejected"]), "nilai", report["total_value"])
    print("Sampel kode kelas diterima:", [r.request_id for r in report["bookings"][:5]])
print("Baris Excel dikecualikan:", len(feb_allocation_demo["excluded"]))


Integrasi fixture: pasangan 15+20=35, DP=19, slot unik F802=2.
SIMULASI EXCEL: Simulasi mulai ruang kosong; kapasitas asumsi Excel, fasilitas belum diketahui, pasangan belum resmi tidak digabung.
Minggu (2026, 41) diterima 189 ditolak 7 nilai 1890
Sampel kode kelas diterima: ['3416/BRKM', '3730/ARK', '2509/ARK', '2510/ARL', '2504/ARL']
Baris Excel dikecualikan: 47


## 7. Pengujian Lengkap — Semua anggota

T01–T18 sudah diimplementasikan dan didaftarkan pada TEST_RUNNERS. RUN_TESTS=True
menjalankan seluruhnya; kolom status berasal dari eksekusi assertion, bukan diisi
manual. Pengujian memakai fixture terkontrol, terpisah dari data aplikasi Excel.

| ID | Input/kondisi | Hasil yang harus diperiksa |
|---|---|---|
| T01 | Campuran empat kategori | Ujian, praktikum, kuliah, event; dalam kategori jam naik |
| T02 | Dua ujian, jam sama | Urutan masuk dipertahankan |
| T03 | Pop antrean kosong | IndexError yang ditangani |
| T04 | Put/get 10 ribu ID unik | Semua data kembali benar |
| T05 | F708 dan F807 pakai hash penjumlahan/8 bucket | Kedua key di chain tetap terbaca |
| T06 | Load factor melewati 0,7 | Ukuran berubah, semua key tetap ada |
| T07 | Prefix ak | ACC13101 dan ACC13102 |
| T08 | Prefix tidak ditemukan | Daftar kosong |
| T09 | Prefix Dasar + tabel pasangan | MGT12119 dan MGT92119 |
| T10 | Insert naik/turun | RR/LL; tinggi dan keseimbangan benar |
| T11 | Insert zig-zag | LR/RL; tinggi dan keseimbangan benar |
| T12 | Peserta 35 | F705, kapasitas 40; peserta 101 → None |
| T13 | 100 kasus acak n≤15 | Nilai DP sama dengan semua kombinasi; hasil valid |
| T14 | Kegiatan tidak bertabrakan, nilai positif | Semua terpilih |
| T15 | Semua saling bertabrakan | Salah satu bernilai paling besar |
| T16 | PQ → pemilihan jadwal → update | Data ruang dan kedua indeks tetap konsisten |
| T17 | Kegiatan ditolak lalu dicoba ruang lain | Diproses sekali per ruang; berhenti saat ruang habis |
| T18 | Gabung pasangan kelas 15+20 | Peserta 35≤40; kelebihan kapasitas ditolak |

Tambahan yang dijalankan: aging/FIFO dan build Heap, 1.000 operasi acak,
Hash update/delete/rehash, alias Trie, seluruh rotasi AVL, delete 200 key acak,
payload kedua indeks, kegagalan transaksi yang disengaja, slot unik,
booking awal, fasilitas eksplisit, ID berulang, input kosong, dan batas minggu.

Kasus DP acak n≤15 dibandingkan dengan brute force pada 100 seed/input yang
terkontrol. Uji retry memeriksa pasangan request–ruang tidak berulang dan berhenti
ketika ruang habis. Uji gabungan mempertahankan source_ids dan menolak kapasitas
berlebih. Kasus gagal menguji penolakan yang diharapkan; tidak dilaporkan sebagai
booking berhasil.


In [20]:
def test_t07():
    trie = Trie()
    for code, name in [("ACC13101", "Akuntansi Biaya"), ("ACC13102", "Akuntansi Manajerial")]:
        trie.insert(name, code)
    trie.insert("AKUNTANSI BIAYA", "ACC13101")
    assert set(trie.autocomplete(" ak ")) == {"ACC13101", "ACC13102"}
    assert len(trie.autocomplete("ak")) == 2


def test_t08():
    trie = Trie()
    trie.insert("Akuntansi", "ACC")
    assert trie.autocomplete("tidak-ada") == []
    assert trie.search("Akun") == []
    assert trie.search("Akuntansi") == ["ACC"]


def test_t09():
    trie = Trie()
    trie.insert("Dasar-Dasar Manajemen", "MGT12119")
    eq, catalog = HashTableChaining(), HashTableChaining()
    eq.put("MGT12119", "MGT92119")
    eq.put("MGT92119", "MGT12119")
    for code in ["MGT12119", "MGT92119"]:
        catalog.put(code, True)
    assert set(bilingual_autocomplete(trie, "Dasar", eq, catalog)) == {"MGT12119", "MGT92119"}


def test_rotation_sequence(values):
    tree = AVLTree()
    for value in values:
        tree.insert(Room(f"R-{value}", value))
        assert tree.validate()
    assert tree.root.key[0] == 20 and tree.rotations > 0


def test_t10():
    test_rotation_sequence([10, 20, 30])
    test_rotation_sequence([30, 20, 10])


def test_t11():
    test_rotation_sequence([30, 10, 20])
    test_rotation_sequence([10, 30, 20])


def test_t12():
    tree = AVLTree()
    for room in [Room("F708", 30), Room("F705", 40), Room("F701", 45), Room("A", 40), Room("BIG", 100)]:
        tree.insert(room)
    assert tree.ceiling_capacity(35).room_id == "A"  # Tie dipecahkan room_id.
    assert tree.ceiling_capacity(101) is None
    assert tree.validate()
    original = AVLTree()
    for room in rooms:
        original.insert(room)
    assert original.ceiling_capacity(35).room_id == "F705"


def test_t13():
    assert verify_dp_random_cases(100, SEED).startswith("100 kasus")


def test_t14():
    batch = [heap_test_request(str(i), start=435 + 60 * i) for i in range(4)]
    assert optimize_schedule(batch) == (40, ["0", "1", "2", "3"])
    assert optimize_schedule([]) == (0, [])
    assert no_overlap(batch)


def test_t15():
    batch = [replace(heap_test_request(str(i)), value=i + 1) for i in range(5)]
    assert optimize_schedule(batch) == (5, ["4"])
    for invalid in [[batch[0], batch[0]], [batch[0], replace(batch[1], room_id="OTHER")],
                    [batch[0], replace(batch[1], day="2026-10-09")]]:
        try:
            optimize_schedule(invalid)
        except ValueError:
            continue
        raise AssertionError("DP harus menolak ID berulang/lintas ruang atau hari.")


def test_t16():
    registry = make_room_registry([Room("F802", 60)])
    report = schedule_across_rooms(requests, *registry, allow_alternative_rooms=False)
    assert report["priority_order"] == ["J3", "J5", "J2", "J6", "J1", "J4"]
    assert report["total_value"] == 19
    assert {r.request_id for r in report["bookings"]} == {"J3", "J4", "J5"}
    assert registry[0].get("F802").used_slots == 2  # Bukan +=3 kegiatan.
    assert validate_room_registry(registry[0], registry[2], registry[1])
    assert len(report["bookings"]) + len(report["rejected"]) == len(requests)


def test_t17():
    batch = [replace(heap_test_request("A"), room_id="R1"),
             replace(heap_test_request("B"), room_id="R1")]
    registry = make_room_registry([Room("R1", 30), Room("R2", 40)])
    report = schedule_across_rooms(batch, *registry)
    assert len(report["bookings"]) == 2 and not report["rejected"]
    pairs = [(a["request_id"], a["room_id"]) for a in report["attempts"]]
    assert len(pairs) == len(set(pairs)) <= 4
    # Ruang habis: tetap berhenti dengan laporan penolakan, bukan loop antrean.
    third = replace(heap_test_request("C"), room_id="R1")
    registry = make_room_registry([Room("R1", 30), Room("R2", 40)])
    report = schedule_across_rooms(batch + [third], *registry)
    assert len(report["bookings"]) == 2 and len(report["rejected"]) == 1
    assert len(report["attempts"]) <= 6
    assert sum(len(ids) for _, ids in report["tried_rooms"].items()) <= 6


def test_t18():
    table, _, eq, _ = build_indexes()
    merged = merge_equivalent_classes(class_records, eq, table)
    assert len(merged) == 1 and merged[0].participants == 35
    assert set(merged[0].source_ids) == {"DEMO-K01", "DEMO-K02"}
    assert merge_equivalent_classes(merged, eq, table) == merged
    try:
        merge_equivalent_classes([class_records[0], replace(class_records[1], participants=30)], eq, table)
    except ValueError:
        pass
    else:
        raise AssertionError("Gabungan 45 peserta melebihi kapasitas 40.")
    for changed in [replace(class_records[1], lecturer_id="OTHER"),
                    replace(class_records[1], day="2026-10-09"),
                    replace(class_records[1], start=500, end=600),
                    replace(class_records[1], activity_type="Event")]:
        assert len(merge_equivalent_classes([class_records[0], changed], eq, table)) == 2


def test_t19():
    rng = Random(SEED)
    for case in range(10):
        sample = generate_requests(rng.randrange(0, 16), seed=rng.randrange(1_000_000))
        expected = optimize_schedule(sample)
        memo_value, memo_ids, memo_calls = optimize_schedule_memo(sample)
        assert (memo_value, memo_ids) == expected, "Memoisasi harus identik dengan tabulasi."
        naive_value, naive_calls = naive_recursive(sample)
        assert naive_value == memo_value, "Nilai optimum naif harus sama."
        assert memo_calls <= naive_calls, "Memoisasi tidak boleh lebih boros pemanggilan dari naif."
    chain = chain_requests(12)
    assert optimize_schedule_memo(chain)[:2] == optimize_schedule(chain)
    assert naive_recursive(chain) == (5 * 12, 2 ** 13 - 1)  # Rantai: eksponensial murni.
    assert optimize_schedule_memo([]) == (0, [], 0) and naive_recursive([]) == (0, 0)


def test_avl_delete_and_atomic_updates():
    rng = Random(SEED)
    tree = AVLTree()
    values = list(range(1, 201))
    rng.shuffle(values)
    for value in values:
        tree.insert(Room(f"R-{value}", value))
    rng.shuffle(values)
    remaining = set(values)
    for value in values:
        assert tree.delete((value, f"R-{value}"))
        remaining.remove(value)
        assert tree.validate()
        assert {r.capacity for r in tree.inorder()} == remaining
        assert not tree.delete((value, f"R-{value}"))
    assert tree.root is None
    registry = make_room_registry([Room("R", 40, used_slots=9)])
    old_key = (9 / 15, "R")
    update_room_usage(registry[0], registry[2], "R", 10, registry[1])
    assert registry[0].get("R").used_slots == 10
    assert old_key not in [(r.utilization, r.room_id) for r in registry[2].inorder()]
    assert validate_room_registry(registry[0], registry[2], registry[1])
    for invalid in [-1, 16, True, 1.5]:
        before = (list(registry[0].items()), registry[1].root, registry[2].root)
        try:
            update_room_usage(registry[0], registry[2], "R", invalid, registry[1])
        except ValueError:
            assert before == (list(registry[0].items()), registry[1].root, registry[2].root)
        else:
            raise AssertionError("Update tidak valid harus ditolak tanpa mutasi.")
    # Simulasikan kegagalan insert pada salinan untuk memeriksa transaksi.
    class FailingUsageAVL(AVLTree):
        def insert(self, key, room=None):
            target = key if room is None else room
            if target.used_slots == 11:
                raise RuntimeError("Kegagalan uji yang disengaja")
            return super().insert(key, room)
    bad = FailingUsageAVL("utilization")
    bad.insert(registry[0].get("R"))
    before = (registry[0].get("R"), registry[1].root, bad.root)
    try:
        update_room_usage(registry[0], bad, "R", 11, registry[1])
    except RuntimeError:
        assert before == (registry[0].get("R"), registry[1].root, bad.root)
    else:
        raise AssertionError("Uji kegagalan tidak berjalan.")


def test_integration_boundaries():
    first = replace(heap_test_request("A", start=435), end=495, room_id="R")
    adjacent = replace(first, request_id="B", start=495, end=555)
    registry = make_room_registry([Room("R", 40)])
    report = schedule_across_rooms([first, adjacent], *registry)
    assert len(report["bookings"]) == 2 and registry[0].get("R").used_slots == 1
    # Existing exact interval permits a non-overlapping booking in the same slot.
    registry = make_room_registry([Room("R", 40, used_slots=1)])
    report = schedule_across_rooms([adjacent], *registry, existing_bookings=[first])
    assert len(report["bookings"]) == 1 and registry[0].get("R").used_slots == 1
    # An unknown interval in an occupied slot is conservatively blocked.
    registry = make_room_registry([Room("R", 40, used_slots=1)])
    report = schedule_across_rooms([adjacent], *registry,
        occupied_slots={"R": {(first.day, 0)}})
    assert not report["bookings"] and len(report["rejected"]) == 1
    for batch in [[first, first], [first, replace(adjacent, day="2026-10-19")]]:
        registry = make_room_registry([Room("R", 40)])
        try:
            schedule_across_rooms(batch, *registry)
        except ValueError:
            assert registry[0].get("R").used_slots == 0
        else:
            raise AssertionError("ID berulang/lintas minggu harus ditolak.")
    registry = make_room_registry([Room("R", 40)])
    report = schedule_across_rooms([first], *registry, required_room_types={"A": "Laboratory"})
    assert not report["bookings"] and "fasilitas" in report["rejected"][0]["reason"]
    empty = schedule_across_rooms([], *make_room_registry([]))
    assert empty["bookings"] == [] and empty["rejected"] == []


TEST_IDS = [f"T{i:02d}" for i in range(1, 20)]
TEST_RUNNERS = {"T01": test_t01, "T02": test_t02, "T03": test_t03,
                "T04": test_t04, "T05": test_t05, "T06": test_t06,
                "T07": test_t07, "T08": test_t08, "T09": test_t09,
                "T10": test_t10, "T11": test_t11, "T12": test_t12,
                "T13": test_t13, "T14": test_t14, "T15": test_t15,
                "T16": test_t16, "T17": test_t17, "T18": test_t18, "T19": test_t19}
RUN_TESTS = True


def run_test_plan():
    report = []
    for test_id in TEST_IDS:
        action = TEST_RUNNERS.get(test_id)
        if action is None:
            report.append((test_id, "BELUM DIISI", "Fungsi uji belum didaftarkan."))
            continue
        try:
            action()
        except NotImplementedError:
            report.append((test_id, "BELUM DIIMPLEMENTASIKAN", "Algoritma masih TODO."))
        except Exception as error:
            report.append((test_id, "GAGAL", f"{type(error).__name__}: {error}"))
        else:
            report.append((test_id, "LULUS", ""))
    return report


if RUN_TESTS:
    test_report = run_test_plan()
    for row in test_report:
        print(*row, sep=" | ")
    assert all(status == "LULUS" for _, status, _ in test_report), "Periksa pengujian yang gagal."
    test_avl_delete_and_atomic_updates()
    test_integration_boundaries()
    print("Tambahan: delete AVL acak, transaksi gagal, slot unik, booking awal, fasilitas, batas minggu: LULUS.")


T01 | LULUS | 
T02 | LULUS | 
T03 | LULUS | 
T04 | LULUS | 
T05 | LULUS | 
T06 | LULUS | 
T07 | LULUS | 
T08 | LULUS | 
T09 | LULUS | 
T10 | LULUS | 
T11 | LULUS | 
T12 | LULUS | 
T13 | LULUS | 
T14 | LULUS | 
T15 | LULUS | 
T16 | LULUS | 
T17 | LULUS | 
T18 | LULUS | 
Tambahan: delete AVL acak, transaksi gagal, slot unik, booking awal, fasilitas, batas minggu: LULUS.


## 8. Percobaan Kecepatan — Wildan dan tim

**Rangkaian benchmark sudah diimplementasikan dan dijalankan.** Ada 58 baris hasil
aktual untuk 240, 1.000, 10.000, dan 100.000 input, ditambah pembanding DP kecil.
Semua menggunakan data **sintetis**, bukan Excel FEB. Angka 240 adalah ukuran
percobaan; jumlah baris Excel tetap dihitung dari berkas impor.

| Percobaan | Pembanding dan kendali |
|---|---|
| Heap build | Heapify diukur sendiri; tidak dibandingkan dengan satu pop |
| Heap push N | Sorted-list insertion untuk N input yang sama |
| Heap push-pop | Maksimal 100 pasangan operasi pada antrean awal berukuran N yang sama |
| Hash get | 100 key yang sama, 50% ditemukan; pembanding linear scan |
| Trie sparse | Prefix sama, satu hasil pada semua ukuran; pembanding startswith |
| Trie dense | Prefix banyak hasil hanya N≤1.000; biaya output/deduplikasi dicatat |
| AVL update + top-k | Delete/insert satu key dan k=10; list melakukan update+sort+ambil k |
| DP skala besar | Berdiri sendiri; tidak dibandingkan langsung dengan BF input kecil |
| DP vs BF | Input identik N=8,12,15; nilai hasil harus sama sebelum diukur |

Setiap waktu adalah **median 5 pengulangan** dengan seed 42. State mutable dibuat
ulang di luar timer; struktur read-only boleh dipakai ulang dengan counter direset.
Setup dan pembacaan counter berada di luar timer. Hasil menyimpan waktu setiap run,
jumlah perbandingan Heap, probe Hash, dan rotasi AVL yang benar-benar dihasilkan
operasi. Tidak ada counter Trie yang direka.

`tracemalloc` dijalankan terpisah untuk **alokasi Python tambahan selama action**.
Angka `peak_bytes` bukan seluruh RAM proses, bukan memori struktur yang sudah
dibangun di setup, dan tidak mencakup seluruh alokasi native. Untuk Heap build,
pembangunan memang berada di action sehingga biaya pembangunannya ikut terukur.

Sel menulis `smart_campus_benchmark.csv`, metadata lingkungan `.json`, grafik
`.html`, dan spesifikasi `.vl.json`. Grafik dikelompokkan per operasi; sumbu log
dan skala vertikal independen tidak boleh dibaca sebagai peringkat seluruh algoritma.
HTML memerlukan internet untuk memuat pustaka grafik. Pada Colab grafik juga
ditampilkan inline jika IPython tersedia.

Output tersimpan diverifikasi dengan Python lokal, **bukan klaim pengukuran pada
mesin Colab**. Versi Python, arsitektur, sistem operasi, waktu UTC, seed, jumlah run,
dan fingerprint input dicatat di `benchmark_environment`. Jalankan ulang di Colab
untuk memperoleh angka mesin tersebut. Implementasi Python manual bisa kalah dari
list bawaan yang operasinya memakai C; hasil tidak dipaksa sesuai ekspektasi teori.

`RUN_BENCHMARKS=True` menjalankan seluruh rentang. Pengguna dapat menggantinya ke
False untuk melewati pengukuran saat sedang memperbaiki kode. Pemanggilan
`run_benchmarks(sizes=[240], repeat=5)` tersedia untuk percobaan singkat.

UI Streamlit dasar sudah tersedia sebagai proyek terpisah. Pekerjaan pada bagian
ini menyediakan layanan simulasi dan kontrak hasil; sinkronisasi kontrol booking
dan dashboard benchmark UI merupakan pekerjaan terpisah dari penyelesaian Colab ini.


In [21]:
from bisect import insort
import gc
import hashlib
import platform
import sys
from datetime import datetime, timezone


def measure_operation(setup, action, repeat=5, read_metrics=None):
    if type(repeat) is not int or repeat < 1:
        raise ValueError("Pengulangan harus bilangan bulat positif.")
    durations, metrics = [], []
    for _ in range(repeat):
        state = setup()  # State mutable baru; di luar waktu operasi.
        started = perf_counter()
        result = action(state)
        durations.append(perf_counter() - started)
        if read_metrics is not None:
            metrics.append(read_metrics(state, result))
        del state, result
    return {"median_seconds": median(durations), "runs": repeat,
            "run_seconds": durations, "metrics": metrics}


def measure_peak_memory(setup, action):
    state = setup()  # Memori state awal tidak masuk pengukuran action.
    gc.collect()
    tracemalloc.start()
    try:
        result = action(state)
        _, peak = tracemalloc.get_traced_memory()
        del result
        return peak
    finally:
        tracemalloc.stop()


def benchmark_heap_setup(batch, built=False):
    queue = MinHeapPriorityQueue()
    if built:
        queue.build(batch)
        queue.comparisons = 0
    return queue


def benchmark_sorted_insert(batch, values):
    for key, request in values:
        insort(batch, (key, request))


def benchmark_heap_insert(queue, values):
    for request in values:
        queue.push(request)


def benchmark_heap_cycle(queue, values):
    for request in values:
        queue.push(request)
        queue.pop()


def benchmark_list_cycle(batch, values):
    for key, request in values:
        insort(batch, (key, request))
        batch.pop(0)


def benchmark_hash_lookup(table, queries):
    return [table.get(key) for key in queries]


def benchmark_linear_lookup(pairs, queries):
    return [next((value for key, value in pairs if key == query), None) for query in queries]


def benchmark_avl_setup(room_list):
    tree = AVLTree("utilization")
    for room in room_list:
        tree.insert(room)
    tree.rotations = 0
    return tree


def benchmark_avl_update(tree, old, new, k):
    assert tree.delete((old.utilization, old.room_id))
    tree.insert(new)
    return tree.top_k_utilization(k)


def benchmark_list_update(room_list, old, new, k):
    index = next(i for i, room in enumerate(room_list) if room.room_id == old.room_id)
    room_list[index] = new
    return sorted(room_list, key=lambda room: (room.utilization, room.room_id), reverse=True)[:k]


def run_benchmarks(sizes=(240, 1000, 10000, 100000), repeat=5, seed=SEED):
    if not sizes or any(type(n) is not int or n < 1 for n in sizes):
        raise ValueError("Ukuran benchmark harus bilangan bulat positif.")
    results = []
    def record(module, operation, implementation, n, k, setup, action, metric=None, query=None):
        timing = measure_operation(setup, action, repeat, metric)
        peak = measure_peak_memory(setup, action)
        row = {"module": module, "operation": operation, "implementation": implementation,
               "n": n, "k": k, "source": "synthetic", "seed": seed,
               "median_seconds": timing["median_seconds"], "peak_bytes": peak,
               "runs": repeat, "query": query,
               "run_seconds": timing["run_seconds"], "operation_metrics": timing["metrics"],
               "memory_scope": "allocation during action; setup excluded"}
        results.append(row)
        print(f"Benchmark {module}/{operation}/{implementation} n={n}: "
              f"{row['median_seconds']:.6f}s; peak action={peak} B", flush=True)

    for n in sizes:
        batch = generate_requests(n, seed)
        keyed = [((request.priority, request.start, i), request) for i, request in enumerate(batch)]
        extra = [replace(batch[i % n], request_id=f"EXTRA-{i}") for i in range(min(100, n))]
        extra_keyed = [((request.priority, request.start, n + i), request) for i, request in enumerate(extra)]
        metric = lambda queue, result: {"comparisons": queue.comparisons}
        record("Heap/PQ", "build", "manual heapify", n, n,
               MinHeapPriorityQueue, lambda queue: queue.build(batch), metric)
        record("Heap/PQ", "push n", "manual heap", n, n,
               MinHeapPriorityQueue, lambda queue: benchmark_heap_insert(queue, batch), metric)
        record("Heap/PQ", "push n", "sorted list", n, n,
               list, lambda values: benchmark_sorted_insert(values, keyed))
        record("Heap/PQ", "push-pop pairs", "manual heap", n, len(extra),
               lambda: benchmark_heap_setup(batch, True), lambda queue: benchmark_heap_cycle(queue, extra), metric)
        record("Heap/PQ", "push-pop pairs", "sorted list", n, len(extra),
               lambda: sorted(keyed), lambda values: benchmark_list_cycle(values, extra_keyed))

        pairs = [(f"ID-{i:06d}", i) for i in range(n)]
        table = HashTableChaining()
        for key, value in pairs:
            table.put(key, value)
        queries = [f"ID-{(i * 7919) % n:06d}" if i % 2 == 0 else f"MISSING-{i}" for i in range(100)]
        assert benchmark_hash_lookup(table, queries) == benchmark_linear_lookup(pairs, queries)
        def hash_setup():
            table.probes = 0
            return table  # Struktur read-only pada get; reset penghitung tiap run.
        record("Hash", "get 50% hit", "chaining", n, len(queries), hash_setup,
               lambda state: benchmark_hash_lookup(state, queries),
               lambda state, result: {"probes": state.probes})
        record("Hash", "get 50% hit", "linear scan", n, len(queries), lambda: pairs,
               lambda state: benchmark_linear_lookup(state, queries))

        entries = generate_search_entries(n)
        names = Trie()
        for entry in entries:
            names.insert(entry.name, entry.entry_id)
        # Satu hasil pada semua ukuran agar biaya output sebanding.
        for operation, prefix in [("prefix sparse", entries[-1].name.casefold())] + (
                [("prefix dense", "akuntansi ")] if n <= 1000 else []):
            expected = [entry.entry_id for entry in entries if entry.name.casefold().startswith(prefix)]
            assert set(names.autocomplete(prefix)) == set(expected)
            record("Trie", operation, "manual trie", n, len(expected), lambda: names,
                   lambda state: state.autocomplete(prefix), query=prefix)
            record("Trie", operation, "linear startswith", n, len(expected), lambda: entries,
                   lambda state: [entry.entry_id for entry in state if entry.name.casefold().startswith(prefix)], query=prefix)
        del names, entries
        gc.collect()

        room_list = generate_rooms(n, seed)
        old = room_list[-1]
        new = replace(old, used_slots=(old.used_slots + 1) % (WEEKLY_SLOTS + 1))
        k = min(10, n)
        expected = benchmark_list_update(list(room_list), old, new, k)
        assert benchmark_avl_update(benchmark_avl_setup(room_list), old, new, k) == expected
        record("AVL", "update + top-k", "manual AVL", n, k,
               lambda: benchmark_avl_setup(room_list), lambda state: benchmark_avl_update(state, old, new, k),
               lambda state, result: {"rotations": state.rotations})
        record("AVL", "update + top-k", "list update + sort", n, k,
               lambda: list(room_list), lambda state: benchmark_list_update(state, old, new, k))
        record("DP", "standalone large", "weighted intervals", n, None,
               lambda: batch, optimize_schedule)

    # Pembanding eksponensial selalu pada batch kecil yang sama, n <= 15.
    for n in (8, 12, 15):
        batch = generate_requests(n, seed)
        assert optimize_schedule(batch)[0] == brute_force_reference(batch)[0]
        record("DP", "same small input", "weighted intervals", n, None, lambda: batch, optimize_schedule)
        record("DP", "same small input", "brute force", n, None, lambda: batch, brute_force_reference)

    # S7: naif vs memo vs tabulasi pada input yang persis sama. Dua jenis input:
    # padat (realistis, banyak tabrakan) dan rantai (p(j)=j-1; kasus terburuk
    # rekursi naif O(2^n), karena itu ukurannya lebih kecil). Metrik "calls"
    # menghitung pemanggilan OPT() untuk naif/memo.
    for operation, sizes, make in (("padat: naif vs memo vs tabulasi", (10, 15, 20, 25), generate_requests),
                                   ("rantai: naif vs memo vs tabulasi", (10, 15, 20), chain_requests)):
        for n in sizes:
            batch = make(n, seed)
            assert naive_recursive(batch)[0] == optimize_schedule_memo(batch)[0] == optimize_schedule(batch)[0]
            record("DP", operation, "rekursi naif", n, None, lambda: batch, naive_recursive,
                   lambda state, result: {"calls": result[1]})
            record("DP", operation, "memoization", n, None, lambda: batch, optimize_schedule_memo,
                   lambda state, result: {"calls": result[2]})
            record("DP", operation, "tabulation", n, None, lambda: batch, optimize_schedule)
    return results


BENCHMARK_SIZES = [240, 1000, 10000, 100000]
RUN_BENCHMARKS = True
benchmark_results = []
benchmark_environment = {"timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0], "system": platform.system(),
    "release": platform.release(), "architecture": platform.machine(),
    "seed": SEED, "repeat": 5,
    "source": "synthetic; not FEB Excel", "sizes": BENCHMARK_SIZES,
    "input_fingerprint": hashlib.sha256(repr(generate_requests(10, SEED)).encode()).hexdigest(),
    "memory_scope": "extra Python allocations during action; setup and native allocations excluded"}
if RUN_BENCHMARKS:
    benchmark_results = run_benchmarks(BENCHMARK_SIZES, repeat=5, seed=SEED)
    benchmark_frame = pd.DataFrame(benchmark_results)
    benchmark_frame.to_csv("smart_campus_benchmark.csv", index=False)
    Path("smart_campus_benchmark_environment.json").write_text(json.dumps(benchmark_environment, indent=2))
    print("Selesai:", len(benchmark_results), "hasil aktual. CSV dan metadata lingkungan tersimpan.")
    print(benchmark_frame[["module", "operation", "implementation", "n", "k", "median_seconds", "peak_bytes"]].to_string(index=False))
    import altair as alt
    chart_data = benchmark_frame.assign(scenario=benchmark_frame["module"] + " · " + benchmark_frame["operation"])
    benchmark_chart = alt.Chart(chart_data).mark_line(point=True).encode(
        x=alt.X("n:Q", scale=alt.Scale(type="log"), title="Jumlah input n (log)"),
        y=alt.Y("median_seconds:Q", scale=alt.Scale(type="log"), title="Median detik (log)"),
        color=alt.Color("implementation:N", title="Implementasi"),
        tooltip=["module", "operation", "implementation", "n", "k", "median_seconds", "peak_bytes"]
    ).properties(width=330, height=190).facet(facet="scenario:N", columns=2).resolve_scale(y="independent")
    benchmark_chart.save("smart_campus_benchmark.html")
    benchmark_chart.save("smart_campus_benchmark.vl.json")
    try:
        from IPython.display import display
    except ImportError:
        print("Grafik tersedia dalam HTML dan spesifikasi Vega-Lite; buka HTML dengan koneksi internet untuk memuat pustaka grafik.")
    else:
        display(benchmark_chart)
else:
    print("Benchmark tidak dijalankan; tidak ada klaim hasil baru.")


Benchmark Heap/PQ/build/manual heapify n=240: 0.000974s; peak action=95700 B
Benchmark Heap/PQ/push n/manual heap n=240: 0.001169s; peak action=95500 B
Benchmark Heap/PQ/push n/sorted list n=240: 0.000059s; peak action=15632 B
Benchmark Heap/PQ/push-pop pairs/manual heap n=240: 0.001602s; peak action=5252 B
Benchmark Heap/PQ/push-pop pairs/sorted list n=240: 0.000030s; peak action=6064 B
Benchmark Hash/get 50% hit/chaining n=240: 0.000072s; peak action=1196 B
Benchmark Hash/get 50% hit/linear scan n=240: 0.000268s; peak action=2072 B
Benchmark Trie/prefix sparse/manual trie n=240: 0.000009s; peak action=777 B
Benchmark Trie/prefix sparse/linear startswith n=240: 0.000014s; peak action=209 B
Benchmark Trie/prefix dense/manual trie n=240: 0.000096s; peak action=4146 B
Benchmark Trie/prefix dense/linear startswith n=240: 0.000016s; peak action=1233 B
Benchmark AVL/update + top-k/manual AVL n=240: 0.000008s; peak action=1088 B
Benchmark AVL/update + top-k/list update + sort n=240: 0.000046

<!DOCTYPE html>

## 9. Ringkasan Biaya Operasi

| Struktur/layanan | Operasi | Waktu | Batas implementasi |
|---|---|---|---|
| Heap/PQ | push/pop; peek | O(log n); O(1) | Hash posisi rata-rata O(1), key pendek |
| Heap/PQ | build; aging seluruh key | O(n); O(n) | Heapify dari bawah, counter FIFO tetap |
| Heap/PQ | pratinjau semua urutan | O(n log n) | Pop pada salinan, ruang O(n) |
| Hash | put/get/delete | Rata-rata O(L), terburuk O(L+n) | L=panjang key; O(1) bila L dibatasi |
| Hash | rehash | O(total panjang key + bucket baru) | Pertumbuhan bucket menjaga load |
| Trie | insert/exact | O(L) rata-rata per jalur | Anak menggunakan Hash tim |
| Trie saat ini | autocomplete | O(p+s+a×k) batas atas | Deduplikasi hasil memakai list membership |
| AVL | insert/delete/ceiling | O(log R) | Delete melakukan rebalance sepanjang jalur balik |
| AVL | top-k | O(log R+k) | Reverse inorder, tanpa sort seluruh ruang |
| DP | persiapan+tabel+traceback | O(n log n), ruang O(n) | Satu batch ruang/hari |
| DP memoization | OPT(j) top-down + cache dict manual | O(n log n), ruang O(n) | Identik dengan tabulasi; rekursi ~n+1 |
| DP rekursi naif | tanpa cache, pembanding | O(2^n) pemanggilan | Dibatasi n kecil (≤26); ukur biaya saja |
| Brute force | semua subset | O(2^n×n) | Hanya N≤15 pada pengujian/benchmark |
| Transaksi registry | salin, validasi, update satu ruang | O(R+bucket) rata-rata, ruang O(R+bucket) | Delete/insert AVL di dalamnya O(log R) |

n=jumlah permintaan/entri, R=jumlah ruang, L=panjang key/nama, p=panjang prefix,
s=jumlah simpul cabang yang dikunjungi, a=referensi ID yang diperiksa,
k=jumlah ID hasil unik. Untuk prefix dengan semua ID unik, komponen deduplikasi
Trie dapat menjadi O(k²). Karena itu benchmark sparse/dense dipisah; biaya
O(p+s+k) pada rancangan ideal bukan jaminan implementasi deduplikasi list ini.

Integrasi bukan satu operasi O(n log n). Selain DP batch, ada daftar calon ruang,
penelusuran riwayat percobaan, pemeriksaan interval booking, serta staging registry.
Batas N×R menyatakan **jumlah percobaan ruang**, bukan batas total operasi dasar.
Kedua indeks AVL dan Hash tetap harus konsisten setelah keputusan.

Tabel kompleksitas menjelaskan pertumbuhan teoretis dengan asumsi yang tercatat.
Angka median benchmark menjelaskan implementasi dan mesin tertentu; satu run atau
satu ukuran tidak membuktikan kompleksitas asimtotik maupun optimum global alokasi.


## 10. Langkah Menuju UAS dan Review

| Tahap | Hasil | Status implementasi notebook |
|---|---|---|
| M1 | Impor/validasi Excel dan generator | Tersedia |
| M1–M2 | Heap/PQ dan Hash | Tersedia, uji T01–T06 |
| M2–M3 | Trie dan dua AVL termasuk delete | Tersedia, uji T07–T12 dan uji delete tambahan |
| M3–M4 | DP dan pembanding semua kombinasi | Tersedia, T13–T15 dan T19 (memoisasi + rekursi naif) |
| M4–M5 | Integrasi, 19 pengujian, eksperimen | Tersedia; hasil aktual dicetak pada bagian 6–8 |
| M6 | Laporan dan latihan demo kelompok | Penjelasan di bawah dapat dipakai untuk latihan |

M1 mengikuti “minggu pertama setelah UTS” pada PPT; tabel ini tidak menetapkan
tanggal UAS atau mengklaim anggota sudah melakukan presentasi.

### Checklist yang dapat diverifikasi melalui kode

- [x] Heap/PQ, Hash, Trie, AVL termasuk delete, dan DP terimplementasi manual.
- [x] Integrasi memakai kontrak Room/ClassRecord/ScheduleRequest yang sama.
- [x] Kapasitas asumsi, bobot kegiatan, waktu tunggu simulasi, dan data sintetis diberi label.
- [x] Hash dan kedua indeks AVL diperiksa konsisten setelah pembaruan.
- [x] T01–T19 memiliki fungsi terdaftar dan hasil eksekusi aktual; fixture DP bernilai 19.
- [x] DP memiliki tabulation, memoization top-down (cache dict manual), dan rekursi naif dengan penghitung pemanggilan.
- [x] Benchmark membandingkan operasi/input setara, median 5 run, memori terpisah.
- [x] Kode kelas/baris Excel dan source_ids gabungan tersedia untuk penelusuran.
- [x] Retry berhenti; strategi beberapa ruang dan batas optimalitas dijelaskan.

### Hal yang memerlukan konfirmasi manusia/data resmi

Kapasitas_asumsi belum kapasitas resmi; pasangan INT–NAS pada Excel belum
dikonfirmasi. Konflik tiga ruang antar-PDF, durasi tertentu, metadata fasilitas,
dan konflik dosen/mahasiswa belum diselesaikan oleh algoritma. ID dosen/pengawas
mengikuti data impor; kecukupan anonimisasi untuk publikasi perlu ditinjau pemilik
data. Laporan/presentasi akhir dan pemahaman setiap anggota tidak diklaim selesai
hanya karena pengujian program lulus.

### Alur penjelasan untuk demo

1. Tunjukkan jumlah dan catatan Excel bagian 0.4; bedakan data resmi/asumsi/fixture.
2. Jelaskan Hash untuk kode lengkap dan Trie untuk awalan/pasangan.
3. Jelaskan PQ sebagai urutan penanganan, AVL sebagai indeks kapasitas/pemakaian,
   dan DP sebagai pemilih kombinasi pada satu ruang/hari.
4. Jalankan bagian 6: fixture 15+20=35, DP=19, serta simulasi berbasis Excel.
   Jelaskan alasan penolakan dan jumlah slot unik, bukan hanya total diterima.
5. Tampilkan T01–T19 dan uji tambahan, kemudian tabel/grafik benchmark beserta
   ukuran, k, jumlah pengulangan, mesin, serta cakupan pengukuran memorinya.
6. Tutup dengan batas model: simulasi tidak menulis jadwal Excel dan bukan optimum
   global, pemeriksaan fasilitas/dosen/mahasiswa masih memerlukan data/aturan.

### Referensi dan adaptasi

Implementasi struktur inti menggunakan kode manual tim; pustaka `heapq` tidak
dipakai sebagai mesin antrean, dan dict tidak menggantikan Hash inti.
`bisect.insort` hanya digunakan sebagai **pembanding sorted list** pada benchmark.
`deepcopy` dipakai untuk staging transaksi, bukan implementasi Hash/AVL pengganti.

| Topik | Rujukan konsep yang dapat ditelusuri |
|---|---|
| Heap/PQ | [Dokumentasi Python — heapq, heap invariant dan priority queue notes](https://docs.python.org/3/library/heapq.html) |
| Hash chaining | [MIT 6.006 — Lecture 08: Hashing with chaining](https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-fall-2011/resources/mit6_006f11_lec08/) |
| Trie/prefix | [Princeton Algorithms 4e — §5.2 Tries](https://algs4.cs.princeton.edu/52trie/) |
| AVL/rotasi | [MIT 6.006 — Lecture 7: Binary Trees, Part 2: AVL](https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-spring-2020/resources/lecture-7-binary-trees-part-2-avl/) |
| Weighted interval scheduling | [Kevin Wayne, Princeton — Dynamic Programming I, weighted interval scheduling](https://www.cs.princeton.edu/~wayne/kleinberg-tardos/pdf/06DynamicProgrammingI.pdf) |
| Dataset dan implementasi kelompok | [Excel FEB](https://drive.google.com/file/d/14ssmxLs2leJDNAUBvTpuk2ukbxm8b-iP/view) dan notebook ini |

Nomor halaman materi kelas S1–S7 tidak direka karena berkas tersebut belum tersedia
pada workspace tugas ini. Referensi di atas adalah sumber konsep tambahan yang
bernama jelas; bukan klaim telah memetakan halaman materi kelas yang belum dibaca.
